# Large-Scale Metaparameter Tuning and Comparison via Multi-Task Bayesian Optimization

This notebook benchmarks and compares two payoff-based zeroth-order Nash Equilibrium seeking algorithms:
1. **Zero-Order Gradient Play (`ZOGrandientPlay`)** with one-point payoff estimation (Tatarenko & Kamgarpour 2022).
2. **Continuous-Time Extremum Seeking (`NESeeking5D`)** with coupled frequency degrees of freedom (Frihauf, Krstic, & Basar 2012; 5D reduction study).

The evaluation is conducted across an ensemble of 100 strongly monotone quadratic games ($N=2, m=1$) with guaranteed interior Nash equilibria using Multi-Task Bayesian Optimization (MTBO, Swersky et al. 2013).

### Scalable Batched Architecture ($4 \times 25$ Batches):
To ensure numerical stability and eliminate exact Gaussian Process memory explosion ($\mathcal{O}(N_{\text{tot}}^2)$ kernel scaling where $25,600 \to >25\text{ GB}$), the 100-game ensemble is partitioned into **4 sequential batches of 25 games each**:
- **Batch 1:** Games 0–24
- **Batch 2:** Games 25–49
- **Batch 3:** Games 50–74
- **Batch 4:** Games 75–99

Each batch executes with ample quasi-random Sobol initialization ($n_{\text{init}} = 16$ for 2D ZOGP, $n_{\text{init}} = 32$ for 5D NESeeking) and strict early stopping criteria (`patience = 10`, `eps_ei = 1e-5`, `tol_stagnation = 1e-5`, `max_bo_steps = 200`, `num_steps = 20000`).

### Option B: Global Consensus Default Determination:
Rather than relying on an unverified arithmetic mean or single-batch default, candidate defaults from each batch ($\hat{\theta}_{\text{def}}^{(1..4)}$) and their component-wise average ($\bar{\theta}_{\text{mean}}$) are pooled into candidate set $\mathcal{C}$ and verified in parallel across all 100 games:
$$\theta_{\text{global\_def}}^* = \arg\min_{\theta \in \mathcal{C}} \frac{1}{100} \sum_{j=1}^{100} \text{Loss}(G_j; \theta)$$
The **Global Tunability Index** across all 100 games is computed against this single universal consensus default:
$$\Delta \text{Loss}_j = \text{Loss}(G_j; \theta_{\text{global\_def}}^*) - \text{Loss}(G_j; \theta_j^*) \ge 0$$

### Notebook Outline:
0. **Cournot Duopoly Preliminaries:** Sanity check, Total Regret minimization experiment, and 6D hyperparameter benchmark.
1. **100-Game Synthetic Ensemble Generation:** Generating 100 games ($\alpha = 0.05$, interior NE) and selecting 5 diverse representative games across the entire ensemble.
2. **Zero-Order Gradient Play ($4 \times 25$ Batched MTBO):** Batched MTBO optimization, Option B consensus default selection, and tailored optima evaluation.
3. **Zero-Order Gradient Play (Optuna Baseline & Visualizations):** 100-trial Optuna baseline and 2x2 action trajectory comparisons on 5 representative games.
4. **Continuous-Time Extremum Seeking (`NESeeking5D` $4 \times 25$ Batched MTBO):** Batched MTBO optimization on 5 continuous dimensions, Option B consensus default selection, and tailored optima evaluation.
5. **Continuous-Time Extremum Seeking (`NESeeking5D` Optuna Baseline & Visualizations):** 100-trial Optuna baseline and 2x2 action trajectory comparisons on 5 representative games.
6. **Quantitative Evaluation & Statistical Reporting:** Paired $t$-test, Wilcoxon signed-rank test, 95% confidence intervals, speedup benchmarks, and automated morning report generation.


## 0. Sanity Check: Frihauf Cournot Duopoly Metaparameter Optimization

This sanity check tunes both **ZOGrandientPlay** and **NESeeking** on the single 2-player Cournot duopoly price game from Frihauf et al. (2012).
- **Stopping Criterion:** `num_steps = 10000` for both optimization and trajectory simulations.
- **Optimization Metric:** Average Euclidean distance of **state history** (not actions) to the true Nash equilibrium $u^* = [43.3333, 36.6667]^T$ over all 10,001 iterations without smoothing window.
- **BO Configuration:** $n_{\text{init}} = 10$, $n_{\text{bo}} = 30$ (40 evaluations per algorithm).
- **Tuned Parameters:**
  - `ZOGrandientPlay`: $v \in [0.5, 10.0]$, $a \in [1.0, 150.0]$ (with $\varepsilon = 10^{-6}$ fixed).
  - `NESeeking`: Player-specific gains $k_{1,2} \in [0.1, 10.0]$, amplitudes $a_{1,2} \in [0.01, 0.2]$, frequencies $\omega_{1,2} \in [15.0, 35.0]$ (with $\Delta t = 0.001$, $\omega_h = 10.0$ fixed).
- **Regret Analysis:** Post-hoc computation of instantaneous and cumulative best-response regret ($R_1(t) = 5(u_1 - u_1^*(u_2))^2$, $R_2(t) = 5(u_2 - u_2^*(u_1))^2$) evaluating the economic exploration cost of the state-optimized parameters.
- **Plotting Y-Axis Limits:** States & Actions limited to $[0, 50]$, Payoffs limited to $[-1000, 1000]$ to remove transient outliers and highlight asymptotic convergence.
- **Artifacts:** Output CSVs, a 12-subplot trajectory plot, a 2-subplot regret dynamics plot, and execution timing summary saved to `data/00_sanity_check/`.


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from payoffnash import (
    EstimateHistory,
    MetaParamOptimization,
    NESeeking,
    QuadraticGame,
    ZOGrandientPlay,
)

# 1. Output directory for sanity check artifacts
sanity_dir = "data/00_sanity_check"
os.makedirs(sanity_dir, exist_ok=True)

# 2. Define the exact Cournot duopoly game from Frihauf et al. / estimator main functions
N = 2
action_sets = [[(0.0, 300.0)], [(0.0, 300.0)]]
Sd, p, m1, m2 = 100.0, 0.2, 30.0, 30.0

def J1(u1, u2):
    s1 = Sd - (u1 - u2) / p
    return float(s1 * (u1 - m1))

def J2(u1, u2):
    s2 = (u1 - u2) / p
    return float(s2 * (u2 - m2))

H = np.array([[-10.0, 5.0], [5.0, -10.0]], dtype=float)
d = np.array([250.0, 150.0], dtype=float)
c = np.array([-3000.0, 0.0], dtype=float)
u_star = np.array([130.0 / 3.0, 110.0 / 3.0], dtype=float)
opt_payoffs = [8000.0 / 9.0, 2000.0 / 9.0]

cournot_game = QuadraticGame(
    N=N,
    action_sets=action_sets,
    payoff_functions=[J1, J2],
    H=H,
    d=d,
    c=c,
    nash_equilibrium=u_star,
    optimal_payoffs=opt_payoffs,
)

# Initial action profile: Interior starting point from Frihauf et al. (2012)
initial_actions = np.array([[50.0], [36.6667]], dtype=float)

# 3. Cost function: Mean distance of state history to true Nash equilibrium over all steps
def state_distance_cost(game: QuadraticGame, history: EstimateHistory) -> float:
    x_star = game.nash_equilibrium.ravel()
    dists = np.linalg.norm(history.state_history - x_star, axis=-1)
    return float(np.mean(dists))

# 4. ZOGrandientPlay MTBO (v, a)
zo_search_space = {
    "v": (0.5, 10.0),
    "a": (1.0, 150.0),
}
zo_fixed_params = {
    "epsilon": 1e-6,
}

print("--- Starting ZOGrandientPlay MTBO (40 evaluations, 10,000 steps each) ---")
zo_opt = MetaParamOptimization(
    game_ensemble=[cournot_game],
    search_space=zo_search_space,
    algorithm_class=ZOGrandientPlay,
    cost_function=state_distance_cost,
    output_path=os.path.join(sanity_dir, "zo_cournot_results.csv"),
    num_steps=10000,
    n_init=10,
    max_bo_steps=30,
    eps_ei=1e-4,
    seed=42,
    fixed_params=zo_fixed_params,
    initial_actions=initial_actions,
)
zo_results = zo_opt.run_optimization_loop()
zo_val_metrics = zo_opt.validate()
zo_best_theta = zo_opt.best_thetas[0]
print(f"ZOGrandientPlay Best Parameters: {zo_results[0]}\n")
print(f"ZOGrandientPlay Surrogate Validation Metrics: {zo_val_metrics}\n")

# 5. NESeeking MTBO (k, a, omega for both players)
ne_search_space = {
    "k[0,0]": (0.1, 10.0),
    "k[1,0]": (0.1, 10.0),
    "a[0,0]": (0.01, 0.2),
    "a[1,0]": (0.01, 0.2),
    "omega[0,0]": (15.0, 35.0),
    "omega[1,0]": (15.0, 35.0),
}
ne_fixed_params = {
    "dt": 0.001,
    "omega_h": np.array([10.0, 10.0]),
}

print("--- Starting NESeeking MTBO (40 evaluations, 10,000 steps each) ---")
ne_opt = MetaParamOptimization(
    game_ensemble=[cournot_game],
    search_space=ne_search_space,
    algorithm_class=NESeeking,
    cost_function=state_distance_cost,
    output_path=os.path.join(sanity_dir, "ne_cournot_results.csv"),
    num_steps=10000,
    n_init=10,
    max_bo_steps=30,
    eps_ei=1e-4,
    seed=42,
    fixed_params=ne_fixed_params,
    initial_actions=initial_actions,
)
ne_results = ne_opt.run_optimization_loop()
ne_val_metrics = ne_opt.validate()
ne_best_theta = ne_opt.best_thetas[0]
print(f"NESeeking Best Parameters: {ne_results[0]}\n")
print(f"NESeeking Surrogate Validation Metrics: {ne_val_metrics}\n")

# 6. Simulate optimal trajectories (num_steps = 10000)
print("--- Simulating optimal trajectories (num_steps = 10000) ---")
zo_best_estimator = zo_opt.build_estimator(zo_best_theta, cournot_game)
ne_best_estimator = ne_opt.build_estimator(ne_best_theta, cournot_game)

t0_zo = time.time()
zo_history = zo_best_estimator.estimate(initial_actions, num_steps=10000)
zo_sim_time = time.time() - t0_zo

t0_ne = time.time()
ne_history = ne_best_estimator.estimate(initial_actions, num_steps=10000)
ne_sim_time = time.time() - t0_ne

# 7. Regret Analysis: Compute instantaneous and cumulative regret profiles
zo_u1 = zo_history.action_history[:, 0]
zo_u2 = zo_history.action_history[:, 1]
zo_r1 = 5.0 * (zo_u1 - (25.0 + 0.5 * zo_u2)) ** 2
zo_r2 = 5.0 * (zo_u2 - (15.0 + 0.5 * zo_u1)) ** 2
zo_tot_regret = zo_r1 + zo_r2
zo_cum_regret = np.cumsum(zo_tot_regret)

ne_u1 = ne_history.action_history[:, 0]
ne_u2 = ne_history.action_history[:, 1]
ne_r1 = 5.0 * (ne_u1 - (25.0 + 0.5 * ne_u2)) ** 2
ne_r2 = 5.0 * (ne_u2 - (15.0 + 0.5 * ne_u1)) ** 2
ne_tot_regret = ne_r1 + ne_r2
ne_cum_regret = np.cumsum(ne_tot_regret)

# 8. Plot 12 Subplots: State, Action, Payoff for both players and both algorithms
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
steps_zo = np.arange(len(zo_history.state_history))
steps_ne = np.arange(len(ne_history.state_history))

# Share x-axis within ZO (rows 0, 1) and within NES (rows 2, 3)
for col in range(3):
    axes[0, col].sharex(axes[1, col])
    axes[2, col].sharex(axes[3, col])

# Row 0: ZO - Player 1
axes[0, 0].plot(steps_zo, zo_history.state_history[:, 0], color="tab:blue", label="State x1")
axes[0, 0].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[0, 0].set_title("ZO: Player 1 - State History")
axes[0, 0].set_ylabel("Price u1")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(steps_zo, zo_history.action_history[:, 0], color="tab:blue", alpha=0.6, label="Action u1")
axes[0, 1].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[0, 1].set_title("ZO: Player 1 - Action History")
axes[0, 1].set_ylabel("Price u1")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(steps_zo, zo_history.payoff_history[:, 0], color="tab:blue", label="Payoff J1")
axes[0, 2].axhline(opt_payoffs[0], color="black", linestyle="--", label="NE J1*")
axes[0, 2].set_title("ZO: Player 1 - Payoff History")
axes[0, 2].set_ylabel("Profit J1")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Row 1: ZO - Player 2
axes[1, 0].plot(steps_zo, zo_history.state_history[:, 1], color="tab:cyan", label="State x2")
axes[1, 0].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[1, 0].set_title("ZO: Player 2 - State History")
axes[1, 0].set_xlabel("Iteration Step (ZO)")
axes[1, 0].set_ylabel("Price u2")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(steps_zo, zo_history.action_history[:, 1], color="tab:cyan", alpha=0.6, label="Action u2")
axes[1, 1].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[1, 1].set_title("ZO: Player 2 - Action History")
axes[1, 1].set_xlabel("Iteration Step (ZO)")
axes[1, 1].set_ylabel("Price u2")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(steps_zo, zo_history.payoff_history[:, 1], color="tab:cyan", label="Payoff J2")
axes[1, 2].axhline(opt_payoffs[1], color="black", linestyle="--", label="NE J2*")
axes[1, 2].set_title("ZO: Player 2 - Payoff History")
axes[1, 2].set_xlabel("Iteration Step (ZO)")
axes[1, 2].set_ylabel("Profit J2")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

# Row 2: NES - Player 1
axes[2, 0].plot(steps_ne, ne_history.state_history[:, 0], color="tab:orange", label="State u_hat1")
axes[2, 0].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[2, 0].set_title("NES: Player 1 - State History")
axes[2, 0].set_ylabel("Price u1")
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(steps_ne, ne_history.action_history[:, 0], color="tab:orange", alpha=0.6, label="Action u1")
axes[2, 1].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[2, 1].set_title("NES: Player 1 - Action History")
axes[2, 1].set_ylabel("Price u1")
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

axes[2, 2].plot(steps_ne, ne_history.payoff_history[:, 0], color="tab:orange", label="Payoff J1")
axes[2, 2].axhline(opt_payoffs[0], color="black", linestyle="--", label="NE J1*")
axes[2, 2].set_title("NES: Player 1 - Payoff History")
axes[2, 2].set_ylabel("Profit J1")
axes[2, 2].legend()
axes[2, 2].grid(True, alpha=0.3)

# Row 3: NES - Player 2
axes[3, 0].plot(steps_ne, ne_history.state_history[:, 1], color="tab:red", label="State u_hat2")
axes[3, 0].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[3, 0].set_title("NES: Player 2 - State History")
axes[3, 0].set_xlabel("Iteration Step (NES)")
axes[3, 0].set_ylabel("Price u2")
axes[3, 0].legend()
axes[3, 0].grid(True, alpha=0.3)

axes[3, 1].plot(steps_ne, ne_history.action_history[:, 1], color="tab:red", alpha=0.6, label="Action u2")
axes[3, 1].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[3, 1].set_title("NES: Player 2 - Action History")
axes[3, 1].set_xlabel("Iteration Step (NES)")
axes[3, 1].set_ylabel("Price u2")
axes[3, 1].legend()
axes[3, 1].grid(True, alpha=0.3)

axes[3, 2].plot(steps_ne, ne_history.payoff_history[:, 1], color="tab:red", label="Payoff J2")
axes[3, 2].axhline(opt_payoffs[1], color="black", linestyle="--", label="NE J2*")
axes[3, 2].set_title("NES: Player 2 - Payoff History")
axes[3, 2].set_xlabel("Iteration Step (NES)")
axes[3, 2].set_ylabel("Profit J2")
axes[3, 2].legend()
axes[3, 2].grid(True, alpha=0.3)

# Apply y-axis limits to prevent initial outliers from masking convergence:
# [0, 50] for actions and states (columns 0 and 1)
# [-1000, 1000] for payoffs (column 2)
for r in range(4):
    axes[r, 0].set_ylim(0, 50)
    axes[r, 1].set_ylim(0, 50)
    axes[r, 2].set_ylim(-1000, 1000)

plt.suptitle("Sanity Check: Optimal Trajectories on Frihauf Cournot Duopoly Game (12 Subplots)", fontsize=16, y=0.995)
plt.tight_layout()
traj_plot_path = os.path.join(sanity_dir, "sanity_check_trajectories.png")
plt.savefig(traj_plot_path, dpi=300, bbox_inches="tight")
print(f"Sanity check trajectory plot saved to: {traj_plot_path}")
plt.show()

# 9. Plot Regret Trajectories: Instantaneous & Cumulative Regret
fig, (ax_inst, ax_cum) = plt.subplots(1, 2, figsize=(14, 5))

ax_inst.plot(steps_zo, zo_tot_regret, color="tab:blue", alpha=0.7, label="ZOGP Instantaneous Regret")
ax_inst.plot(steps_ne, ne_tot_regret, color="tab:orange", alpha=0.7, label="NES Instantaneous Regret")
ax_inst.set_title("Instantaneous Total Regret R1(t) + R2(t)")
ax_inst.set_xlabel("Iteration Step")
ax_inst.set_ylabel("Regret")
ax_inst.set_yscale("log")
ax_inst.grid(True, alpha=0.3)
ax_inst.legend()

ax_cum.plot(steps_zo, zo_cum_regret, color="tab:blue", lw=1.5, label="ZOGP Cumulative Regret")
ax_cum.plot(steps_ne, ne_cum_regret, color="tab:orange", lw=1.5, label="NES Cumulative Regret")
ax_cum.set_title("Cumulative Total Regret Over Time")
ax_cum.set_xlabel("Iteration Step")
ax_cum.set_ylabel("Cumulative Regret")
ax_cum.grid(True, alpha=0.3)
ax_cum.legend()

plt.suptitle("Sanity Check: Total Regret Dynamics Comparison", fontsize=14, y=1.02)
plt.tight_layout()
regret_plot_path = os.path.join(sanity_dir, "cumulative_regret.png")
plt.savefig(regret_plot_path, dpi=300, bbox_inches="tight")
print(f"Sanity check regret plot saved to: {regret_plot_path}")
plt.show()

# 10. Print comprehensive summary
print("=" * 70)
print("SANITY CHECK SUMMARY")
print("=" * 70)
print(f"Nash Equilibrium Reference: u* = [{u_star[0]:.4f}, {u_star[1]:.4f}]")
print(f"Optimal Payoffs Reference:  J* = [{opt_payoffs[0]:.4f}, {opt_payoffs[1]:.4f}]")
print("-" * 70)
print("ZOGrandientPlay:")
print(f"  Best Parameters:     {zo_results[0]}")
print(f"  Optimization Loss:   {zo_results[0]['objective_value']:.4f} (Mean State Dist to NE)")
print(f"  Simulation Time:     {zo_sim_time:.2f} seconds ({len(zo_history.state_history)} steps)")
print(f"  Final State:         u_hat = [{zo_history.state_history[-1][0]:.4f}, {zo_history.state_history[-1][1]:.4f}]")
print(f"  Final Error:         ||u_hat - u*|| = {np.linalg.norm(zo_history.state_history[-1] - u_star):.4f}")
print(f"  Sim Total Regret:    {zo_cum_regret[-1]:.4f} (Mean per step: {np.mean(zo_tot_regret):.4f})")
print("-" * 70)
print("NESeeking:")
print(f"  Best Parameters:     {ne_results[0]}")
print(f"  Optimization Loss:   {ne_results[0]['objective_value']:.4f} (Mean State Dist to NE)")
print(f"  Simulation Time:     {ne_sim_time:.2f} seconds ({len(ne_history.state_history)} steps)")
print(f"  Final State:         u_hat = [{ne_history.state_history[-1][0]:.4f}, {ne_history.state_history[-1][1]:.4f}]")
print(f"  Final Error:         ||u_hat - u*|| = {np.linalg.norm(ne_history.state_history[-1] - u_star):.4f}")
print(f"  Sim Total Regret:    {ne_cum_regret[-1]:.4f} (Mean per step: {np.mean(ne_tot_regret):.4f})")
print("=" * 70)


## 0.1 Experiment: Cournot Duopoly Tuning via Total Regret Minimization

This experiment tunes both **ZOGrandientPlay** and **NESeeking** on the Cournot duopoly price game (Frihauf et al. 2012) using **Total Regret** across both players as the minimization objective in MTBO.

### Mathematical Formulation of Regret:
At each iteration step $t$, given player 2's action $u_2(t)$, player 1's best response action is:
$$u_1^*(u_2(t)) = \arg\max_{u_1 \in [0, 300]} J_1(u_1, u_2(t)) = 25.0 + 0.5 \cdot u_2(t)$$
and player 1's instantaneous regret is the payoff lost relative to the best response:
$$R_1(t) = \max_{u_1' \in [0, 300]} J_1(u_1', u_2(t)) - J_1(u_1(t), u_2(t)) = 5 \cdot \left(u_1(t) - u_1^*(u_2(t))\right)^2 \ge 0$$

Similarly, given player 1's action $u_1(t)$, player 2's best response action is:
$$u_2^*(u_1(t)) = \arg\max_{u_2 \in [0, 300]} J_2(u_1(t), u_2) = 15.0 + 0.5 \cdot u_1(t)$$
and player 2's instantaneous regret is:
$$R_2(t) = \max_{u_2' \in [0, 300]} J_2(u_1(t), u_2') - J_2(u_1(t), u_2(t)) = 5 \cdot \left(u_2(t) - u_2^*(u_1(t))\right)^2 \ge 0$$

The systemic total regret across all iterations for both players is:
$$\text{Total Regret} = \sum_{t=0}^T \left(R_1(t) + R_2(t)\right)$$

### Optimization Objective (Log-Scale Stabilization):
Because exploratory or unstable parameters during Latin Hypercube initialization can incur high regrets ($10^9$) while converged trajectories achieve $\sim 10^4$, raw regret spans 5 orders of magnitude. To ensure well-conditioned Gaussian Process modeling and avoid surrogate variance distortion, MTBO minimizes the strictly monotonic log-scale objective:
$$\text{Cost} = \log_{10}\left(\max\left(\text{Total Regret}, 10^{-12}\right)\right)$$
Since $\arg\min \log_{10}(R) \equiv \arg\min R$, the global optimal parameters are strictly identical while GP predictions remain bounded in $[4, 10]$.

### Configuration:
- **Stopping Criterion:** `num_steps = 10000` for both optimization and trajectory simulations (40 BO evaluations per algorithm: $n_{\text{init}}=10, n_{\text{bo}}=30$).
- **Plot Limits:** $[0, 50]$ for states & actions, $[-1000, 1000]$ for payoffs.
- **Output Artifacts:** Saved to `data/01_total_regret_test/`.


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from payoffnash import (
    EstimateHistory,
    MetaParamOptimization,
    NESeeking,
    QuadraticGame,
    ZOGrandientPlay,
)

# 1. Output directory for total regret test artifacts
regret_dir = "data/01_total_regret_test"
os.makedirs(regret_dir, exist_ok=True)

# 2. Define the exact Cournot duopoly game from Frihauf et al. (2012)
N = 2
action_sets = [[(0.0, 300.0)], [(0.0, 300.0)]]
Sd, p, m1, m2 = 100.0, 0.2, 30.0, 30.0

def J1(u1, u2):
    s1 = Sd - (u1 - u2) / p
    return float(s1 * (u1 - m1))

def J2(u1, u2):
    s2 = (u1 - u2) / p
    return float(s2 * (u2 - m2))

H = np.array([[-10.0, 5.0], [5.0, -10.0]], dtype=float)
d = np.array([250.0, 150.0], dtype=float)
c = np.array([-3000.0, 0.0], dtype=float)
u_star = np.array([130.0 / 3.0, 110.0 / 3.0], dtype=float)
opt_payoffs = [8000.0 / 9.0, 2000.0 / 9.0]

cournot_game = QuadraticGame(
    N=N,
    action_sets=action_sets,
    payoff_functions=[J1, J2],
    H=H,
    d=d,
    c=c,
    nash_equilibrium=u_star,
    optimal_payoffs=opt_payoffs,
)

initial_actions = np.array([[50.0], [36.6667]], dtype=float)

# 3. Cost function: Log10 Total Regret across both players over all steps
def total_regret_cost(game: QuadraticGame, history: EstimateHistory) -> float:
    """
    Computes log10 of total systemic regret across both players over all steps.
    Using log10 stabilizes the Gaussian Process in MTBO, preventing scale distortion
    from initial transients spanning multiple orders of magnitude (10^4 to 10^9).
    """
    u1 = history.action_history[:, 0]
    u2 = history.action_history[:, 1]
    br1 = 25.0 + 0.5 * u2
    br2 = 15.0 + 0.5 * u1
    regret1 = 5.0 * (u1 - br1) ** 2
    regret2 = 5.0 * (u2 - br2) ** 2
    tot = float(np.sum(regret1 + regret2))
    return float(np.log10(max(tot, 1e-12)))

# 4. ZOGrandientPlay MTBO (Minimizing Log10 Total Regret)
zo_search_space = {
    "v": (0.5, 10.0),
    "a": (1.0, 150.0),
}
zo_fixed_params = {
    "epsilon": 1e-6,
}

print("--- Starting ZOGrandientPlay MTBO (Log10 Total Regret, 40 evaluations, 10,000 steps each) ---")
zo_opt = MetaParamOptimization(
    game_ensemble=[cournot_game],
    search_space=zo_search_space,
    algorithm_class=ZOGrandientPlay,
    cost_function=total_regret_cost,
    output_path=os.path.join(regret_dir, "zo_cournot_regret_results.csv"),
    num_steps=10000,
    n_init=10,
    max_bo_steps=30,
    eps_ei=1e-4,
    seed=42,
    fixed_params=zo_fixed_params,
    initial_actions=initial_actions,
)
zo_results = zo_opt.run_optimization_loop()
zo_val_metrics = zo_opt.validate()
zo_best_theta = zo_opt.best_thetas[0]
print(f"ZOGrandientPlay Best Parameters: {zo_results[0]}\n")
print(f"ZOGrandientPlay Surrogate Validation Metrics: {zo_val_metrics}\n")

# 5. NESeeking MTBO (Minimizing Log10 Total Regret)
ne_search_space = {
    "k[0,0]": (0.1, 10.0),
    "k[1,0]": (0.1, 10.0),
    "a[0,0]": (0.01, 0.2),
    "a[1,0]": (0.01, 0.2),
    "omega[0,0]": (15.0, 35.0),
    "omega[1,0]": (15.0, 35.0),
}
ne_fixed_params = {
    "dt": 0.001,
    "omega_h": np.array([10.0, 10.0]),
}

print("--- Starting NESeeking MTBO (Log10 Total Regret, 40 evaluations, 10,000 steps each) ---")
ne_opt = MetaParamOptimization(
    game_ensemble=[cournot_game],
    search_space=ne_search_space,
    algorithm_class=NESeeking,
    cost_function=total_regret_cost,
    output_path=os.path.join(regret_dir, "ne_cournot_regret_results.csv"),
    num_steps=10000,
    n_init=10,
    max_bo_steps=30,
    eps_ei=1e-4,
    seed=42,
    fixed_params=ne_fixed_params,
    initial_actions=initial_actions,
)
ne_results = ne_opt.run_optimization_loop()
ne_val_metrics = ne_opt.validate()
ne_best_theta = ne_opt.best_thetas[0]
print(f"NESeeking Best Parameters: {ne_results[0]}\n")
print(f"NESeeking Surrogate Validation Metrics: {ne_val_metrics}\n")

# 6. Simulate optimal trajectories (num_steps = 10000)
print("--- Simulating optimal trajectories (num_steps = 10000) ---")
zo_best_estimator = zo_opt.build_estimator(zo_best_theta, cournot_game)
ne_best_estimator = ne_opt.build_estimator(ne_best_theta, cournot_game)

t0_zo = time.time()
zo_history = zo_best_estimator.estimate(initial_actions, num_steps=10000)
zo_sim_time = time.time() - t0_zo

t0_ne = time.time()
ne_history = ne_best_estimator.estimate(initial_actions, num_steps=10000)
ne_sim_time = time.time() - t0_ne

# Compute instantaneous and cumulative regret profiles for visualization
zo_u1 = zo_history.action_history[:, 0]
zo_u2 = zo_history.action_history[:, 1]
zo_r1 = 5.0 * (zo_u1 - (25.0 + 0.5 * zo_u2)) ** 2
zo_r2 = 5.0 * (zo_u2 - (15.0 + 0.5 * zo_u1)) ** 2
zo_tot_regret = zo_r1 + zo_r2
zo_cum_regret = np.cumsum(zo_tot_regret)

ne_u1 = ne_history.action_history[:, 0]
ne_u2 = ne_history.action_history[:, 1]
ne_r1 = 5.0 * (ne_u1 - (25.0 + 0.5 * ne_u2)) ** 2
ne_r2 = 5.0 * (ne_u2 - (15.0 + 0.5 * ne_u1)) ** 2
ne_tot_regret = ne_r1 + ne_r2
ne_cum_regret = np.cumsum(ne_tot_regret)

# 7. Plot 12 Subplots: State, Action, Payoff for both players and both algorithms
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
steps_zo = np.arange(len(zo_history.state_history))
steps_ne = np.arange(len(ne_history.state_history))

# Share x-axis within ZO (rows 0, 1) and within NES (rows 2, 3)
for col in range(3):
    axes[0, col].sharex(axes[1, col])
    axes[2, col].sharex(axes[3, col])

# Row 0: ZO - Player 1
axes[0, 0].plot(steps_zo, zo_history.state_history[:, 0], color="tab:blue", label="State x1")
axes[0, 0].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[0, 0].set_title("ZO: Player 1 - State History")
axes[0, 0].set_ylabel("Price u1")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(steps_zo, zo_history.action_history[:, 0], color="tab:blue", alpha=0.6, label="Action u1")
axes[0, 1].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[0, 1].set_title("ZO: Player 1 - Action History")
axes[0, 1].set_ylabel("Price u1")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(steps_zo, zo_history.payoff_history[:, 0], color="tab:blue", label="Payoff J1")
axes[0, 2].axhline(opt_payoffs[0], color="black", linestyle="--", label="NE J1*")
axes[0, 2].set_title("ZO: Player 1 - Payoff History")
axes[0, 2].set_ylabel("Profit J1")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Row 1: ZO - Player 2
axes[1, 0].plot(steps_zo, zo_history.state_history[:, 1], color="tab:cyan", label="State x2")
axes[1, 0].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[1, 0].set_title("ZO: Player 2 - State History")
axes[1, 0].set_xlabel("Iteration Step (ZO)")
axes[1, 0].set_ylabel("Price u2")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(steps_zo, zo_history.action_history[:, 1], color="tab:cyan", alpha=0.6, label="Action u2")
axes[1, 1].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[1, 1].set_title("ZO: Player 2 - Action History")
axes[1, 1].set_xlabel("Iteration Step (ZO)")
axes[1, 1].set_ylabel("Price u2")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(steps_zo, zo_history.payoff_history[:, 1], color="tab:cyan", label="Payoff J2")
axes[1, 2].axhline(opt_payoffs[1], color="black", linestyle="--", label="NE J2*")
axes[1, 2].set_title("ZO: Player 2 - Payoff History")
axes[1, 2].set_xlabel("Iteration Step (ZO)")
axes[1, 2].set_ylabel("Profit J2")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

# Row 2: NES - Player 1
axes[2, 0].plot(steps_ne, ne_history.state_history[:, 0], color="tab:orange", label="State u_hat1")
axes[2, 0].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[2, 0].set_title("NES: Player 1 - State History")
axes[2, 0].set_ylabel("Price u1")
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(steps_ne, ne_history.action_history[:, 0], color="tab:orange", alpha=0.6, label="Action u1")
axes[2, 1].axhline(u_star[0], color="black", linestyle="--", label="NE x1*")
axes[2, 1].set_title("NES: Player 1 - Action History")
axes[2, 1].set_ylabel("Price u1")
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

axes[2, 2].plot(steps_ne, ne_history.payoff_history[:, 0], color="tab:orange", label="Payoff J1")
axes[2, 2].axhline(opt_payoffs[0], color="black", linestyle="--", label="NE J1*")
axes[2, 2].set_title("NES: Player 1 - Payoff History")
axes[2, 2].set_ylabel("Profit J1")
axes[2, 2].legend()
axes[2, 2].grid(True, alpha=0.3)

# Row 3: NES - Player 2
axes[3, 0].plot(steps_ne, ne_history.state_history[:, 1], color="tab:red", label="State u_hat2")
axes[3, 0].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[3, 0].set_title("NES: Player 2 - State History")
axes[3, 0].set_xlabel("Iteration Step (NES)")
axes[3, 0].set_ylabel("Price u2")
axes[3, 0].legend()
axes[3, 0].grid(True, alpha=0.3)

axes[3, 1].plot(steps_ne, ne_history.action_history[:, 1], color="tab:red", alpha=0.6, label="Action u2")
axes[3, 1].axhline(u_star[1], color="black", linestyle="--", label="NE x2*")
axes[3, 1].set_title("NES: Player 2 - Action History")
axes[3, 1].set_xlabel("Iteration Step (NES)")
axes[3, 1].set_ylabel("Price u2")
axes[3, 1].legend()
axes[3, 1].grid(True, alpha=0.3)

axes[3, 2].plot(steps_ne, ne_history.payoff_history[:, 1], color="tab:red", label="Payoff J2")
axes[3, 2].axhline(opt_payoffs[1], color="black", linestyle="--", label="NE J2*")
axes[3, 2].set_title("NES: Player 2 - Payoff History")
axes[3, 2].set_xlabel("Iteration Step (NES)")
axes[3, 2].set_ylabel("Profit J2")
axes[3, 2].legend()
axes[3, 2].grid(True, alpha=0.3)

# Apply y-axis limits: [0, 50] for states and actions, [-1000, 1000] for payoffs
for r in range(4):
    axes[r, 0].set_ylim(0, 50)
    axes[r, 1].set_ylim(0, 50)
    axes[r, 2].set_ylim(-1000, 1000)

plt.suptitle("Total Regret Test: Optimal Trajectories on Frihauf Cournot Duopoly Game (12 Subplots)", fontsize=16, y=0.995)
plt.tight_layout()
traj_plot_path = os.path.join(regret_dir, "total_regret_trajectories.png")
plt.savefig(traj_plot_path, dpi=300, bbox_inches="tight")
print(f"Trajectory plot saved to: {traj_plot_path}")
plt.show()

# 8. Plot Regret Trajectories: Instantaneous & Cumulative Regret
fig, (ax_inst, ax_cum) = plt.subplots(1, 2, figsize=(14, 5))

ax_inst.plot(steps_zo, zo_tot_regret, color="tab:blue", alpha=0.7, label="ZOGP Instantaneous Regret")
ax_inst.plot(steps_ne, ne_tot_regret, color="tab:orange", alpha=0.7, label="NES Instantaneous Regret")
ax_inst.set_title("Instantaneous Total Regret R1(t) + R2(t)")
ax_inst.set_xlabel("Iteration Step")
ax_inst.set_ylabel("Regret")
ax_inst.set_yscale("log")
ax_inst.grid(True, alpha=0.3)
ax_inst.legend()

ax_cum.plot(steps_zo, zo_cum_regret, color="tab:blue", lw=1.5, label="ZOGP Cumulative Regret")
ax_cum.plot(steps_ne, ne_cum_regret, color="tab:orange", lw=1.5, label="NES Cumulative Regret")
ax_cum.set_title("Cumulative Total Regret Over Time")
ax_cum.set_xlabel("Iteration Step")
ax_cum.set_ylabel("Cumulative Regret")
ax_cum.grid(True, alpha=0.3)
ax_cum.legend()

plt.suptitle("Total Regret Dynamics Comparison", fontsize=14, y=1.02)
plt.tight_layout()
regret_plot_path = os.path.join(regret_dir, "cumulative_regret.png")
plt.savefig(regret_plot_path, dpi=300, bbox_inches="tight")
print(f"Regret dynamics plot saved to: {regret_plot_path}")
plt.show()

# 9. Print comprehensive summary
print("=" * 70)
print("TOTAL REGRET TEST SUMMARY")
print("=" * 70)
print(f"Nash Equilibrium Reference: u* = [{u_star[0]:.4f}, {u_star[1]:.4f}]")
print(f"Optimal Payoffs Reference:  J* = [{opt_payoffs[0]:.4f}, {opt_payoffs[1]:.4f}]")
print("-" * 70)
print("ZOGrandientPlay:")
print(f"  Best Parameters:     {zo_results[0]}")
print(f"  Optimization Loss:   {zo_results[0]['objective_value']:.4f} (log10 Total Regret)")
print(f"  Simulation Time:     {zo_sim_time:.2f} seconds ({len(zo_history.state_history)} steps)")
print(f"  Final State:         u_hat = [{zo_history.state_history[-1][0]:.4f}, {zo_history.state_history[-1][1]:.4f}]")
print(f"  Final Error:         ||u_hat - u*|| = {np.linalg.norm(zo_history.state_history[-1] - u_star):.4f}")
print(f"  Sim Total Regret:    {zo_cum_regret[-1]:.4f} (Mean per step: {np.mean(zo_tot_regret):.4f})")
print("-" * 70)
print("NESeeking:")
print(f"  Best Parameters:     {ne_results[0]}")
print(f"  Optimization Loss:   {ne_results[0]['objective_value']:.4f} (log10 Total Regret)")
print(f"  Simulation Time:     {ne_sim_time:.2f} seconds ({len(ne_history.state_history)} steps)")
print(f"  Final State:         u_hat = [{ne_history.state_history[-1][0]:.4f}, {ne_history.state_history[-1][1]:.4f}]")
print(f"  Final Error:         ||u_hat - u*|| = {np.linalg.norm(ne_history.state_history[-1] - u_star):.4f}")
print(f"  Sim Total Regret:    {ne_cum_regret[-1]:.4f} (Mean per step: {np.mean(ne_tot_regret):.4f})")
print("=" * 70)


## 0.2 Benchmark: 6-Dimensional Hyperparameter Tuning (NESeeking) vs. Optuna Baseline

This benchmark tunes **all 6 continuous hyperparameters** of `NESeeking` on the 2-player Cournot duopoly price game (Frihauf et al. 2012) using **Total Regret** minimization over 10,000 steps ($dt = 0.001$, $\omega_h = 10.0$ fixed) across **512 evaluations**:

### Search Space (6 Continuous Dimensions):
- **Integration Gains:** $k[0,0], k[1,0] \in [0.1, 10.0]$
- **Perturbation Amplitudes:** $a[0,0], a[1,0] \in [0.01, 0.2]$
- **Dither Frequencies:** $\omega[0,0], \omega[1,0] \in [15.0, 35.0]$
- *(Fixed: $dt = 0.001$, $\omega_h = [10.0, 10.0]^T$; $\phi$ eliminated per Frihauf et al. 2012)*

### Optimization Setup (512 Total Evaluations):
1. **Refactored MTBO:** Uses dimension-scaled dynamic Sobol candidate sampling ($N_{\text{cands}} = \min(256 \cdot 6, 4096) = 1536$ candidates per BO step), Swersky Eq. 8 full covariance sum across tasks, batched memory-safe tensor operations, direct task indexing, and safety masking against redundant queries across 64 initial Latin Hypercube samples and 448 BO steps ($n_{\text{init}}=64, n_{\text{bo}}=448$, seed 42).
2. **Baseline Model (Optuna):** Runs an identical 512-evaluation budget (64 startup trials, 448 TPE steps, seed 42) using Optuna's Tree-structured Parzen Estimator (`TPESampler`) on the exact same 10,000-step simulation loss.
3. **Artifacts:** Output CSVs, trajectory plots, and cumulative regret dynamics saved to `data/02_regret_benchmark/`.


In [ ]:
import os
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
import optuna

from payoffnash.game_ensemble import QuadraticGame
from payoffnash.ne_seeking import NESeeking
from payoffnash.mtbo import MetaParamOptimization
from payoffnash.common import EstimateHistory

bench_dir = "data/02_regret_benchmark"
os.makedirs(bench_dir, exist_ok=True)

N = 2
action_sets = [[(0.0, 300.0)], [(0.0, 300.0)]]
Sd, p, m1, m2 = 100.0, 0.2, 30.0, 30.0

def J1(u1, u2):
    s1 = Sd - (u1 - u2) / p
    return float(s1 * (u1 - m1))

def J2(u1, u2):
    s2 = (u1 - u2) / p
    return float(s2 * (u2 - m2))

H = np.array([[-10.0, 5.0], [5.0, -10.0]], dtype=float)
d = np.array([250.0, 150.0], dtype=float)
c = np.array([-3000.0, 0.0], dtype=float)
u_star = np.array([130.0 / 3.0, 110.0 / 3.0], dtype=float)
opt_payoffs = [8000.0 / 9.0, 2000.0 / 9.0]

game = QuadraticGame(
    N=N,
    action_sets=action_sets,
    payoff_functions=[J1, J2],
    H=H,
    d=d,
    c=c,
    nash_equilibrium=u_star,
    optimal_payoffs=opt_payoffs,
)

initial_actions = np.array([[50.0], [36.6667]], dtype=float)

def total_regret_cost(game: QuadraticGame, history: EstimateHistory) -> float:
    u1 = history.action_history[:, 0]
    u2 = history.action_history[:, 1]
    br1 = 25.0 + 0.5 * u2
    br2 = 15.0 + 0.5 * u1
    reg1 = 5.0 * (u1 - br1) ** 2
    reg2 = 5.0 * (u2 - br2) ** 2
    tot = float(np.sum(reg1 + reg2))
    return float(np.log10(max(tot, 1e-12)))

ne_search_space = {
    "k[0,0]": (0.1, 10.0),
    "k[1,0]": (0.1, 10.0),
    "a[0,0]": (0.01, 0.2),
    "a[1,0]": (0.01, 0.2),
    "omega[0,0]": (15.0, 35.0),
    "omega[1,0]": (15.0, 35.0),
}
fixed_params = {
    "dt": 0.001,
    "omega_h": np.array([10.0, 10.0]),
}

TOTAL_EVALS = 512
N_INIT = 64
N_BO = TOTAL_EVALS - N_INIT  # 448

print(f"=== 1. Running Refactored MTBO (6D, {TOTAL_EVALS} total evals: {N_INIT} init + {N_BO} BO, 10,000 steps) ===", flush=True)
t0_mtbo = time.time()
mtbo_opt = MetaParamOptimization(
    game_ensemble=[game],
    search_space=ne_search_space,
    algorithm_class=NESeeking,
    cost_function=total_regret_cost,
    output_path=os.path.join(bench_dir, "mtbo_regret_results.csv"),
    num_steps=10000,
    n_init=N_INIT,
    max_bo_steps=N_BO,
    eps_ei=None,
    seed=42,
    fixed_params=fixed_params,
    initial_actions=initial_actions,
)
mtbo_results = mtbo_opt.run_optimization_loop()
mtbo_val_metrics = mtbo_opt.validate()
t_mtbo = time.time() - t0_mtbo
print(f"MTBO Validation Metrics: {mtbo_val_metrics}\n")
print(f"MTBO completed in {t_mtbo:.2f}s ({t_mtbo/60:.2f} min)!", flush=True)

mtbo_best = mtbo_results[0]
print(f"MTBO Best log10(Regret): {mtbo_best['objective_value']:.4f} (Regret: {10**mtbo_best['objective_value']:.2f})", flush=True)

print(f"\n=== 2. Running Optuna TPE Baseline (6D, {TOTAL_EVALS} trials: {N_INIT} startup + {N_BO} TPE, 10,000 steps) ===", flush=True)
t0_optuna = time.time()

def optuna_objective(trial: optuna.Trial) -> float:
    k0 = trial.suggest_float("k[0,0]", 0.1, 10.0)
    k1 = trial.suggest_float("k[1,0]", 0.1, 10.0)
    a0 = trial.suggest_float("a[0,0]", 0.01, 0.2)
    a1 = trial.suggest_float("a[1,0]", 0.01, 0.2)
    w0 = trial.suggest_float("omega[0,0]", 15.0, 35.0)
    w1 = trial.suggest_float("omega[1,0]", 15.0, 35.0)

    estimator = NESeeking(
        game,
        k=np.array([[k0], [k1]]),
        a=np.array([[a0], [a1]]),
        omega=np.array([[w0], [w1]]),
        omega_h=np.array([10.0, 10.0]),
        dt=0.001,
    )
    res = estimator.estimate(initial_actions, num_steps=10000)
    return total_regret_cost(game, res)

optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler = optuna.samplers.TPESampler(n_startup_trials=N_INIT, seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

step_count = 0
def optuna_callback(study, trial):
    global step_count
    step_count += 1
    if step_count % 50 == 0 or step_count == TOTAL_EVALS:
        print(f"  [Optuna Trial {step_count}/{TOTAL_EVALS}] loss={trial.value:.5f}, best so far={study.best_value:.5f}", flush=True)

study.optimize(optuna_objective, n_trials=TOTAL_EVALS, callbacks=[optuna_callback])
t_optuna = time.time() - t0_optuna
print(f"Optuna completed in {t_optuna:.2f}s ({t_optuna/60:.2f} min)!", flush=True)

optuna_best = study.best_params
optuna_best_val = study.best_value
print(f"Optuna Best log10(Regret): {optuna_best_val:.4f} (Regret: {10**optuna_best_val:.2f})", flush=True)

# Export Optuna trials
optuna_csv_path = os.path.join(bench_dir, "optuna_regret_results.csv")
with open(optuna_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    keys = list(ne_search_space.keys())
    writer.writerow(["trial_id"] + keys + ["objective_value"])
    for trial in study.trials:
        row = [trial.number] + [trial.params[k] for k in keys] + [trial.value]
        writer.writerow(row)

# 5. Full Trajectory Simulation for Head-to-Head Comparison
print("\n=== 3. Simulating Full Trajectories for Head-to-Head Comparison ===", flush=True)
ne_mtbo = mtbo_opt.build_estimator(mtbo_opt.best_thetas[0], game)
res_mtbo = ne_mtbo.estimate(initial_actions, num_steps=10000)

ne_optuna = NESeeking(
    game,
    k=np.array([[optuna_best["k[0,0]"]], [optuna_best["k[1,0]"]]]),
    a=np.array([[optuna_best["a[0,0]"]], [optuna_best["a[1,0]"]]]),
    omega=np.array([[optuna_best["omega[0,0]"]], [optuna_best["omega[1,0]"]]]),
    omega_h=np.array([10.0, 10.0]),
    dt=0.001,
)
res_optuna = ne_optuna.estimate(initial_actions, num_steps=10000)

def compute_regrets(res):
    u1, u2 = res.action_history[:, 0], res.action_history[:, 1]
    br1, br2 = 25.0 + 0.5 * u2, 15.0 + 0.5 * u1
    r1, r2 = 5.0 * (u1 - br1) ** 2, 5.0 * (u2 - br2) ** 2
    inst = r1 + r2
    cum = np.cumsum(inst)
    return inst, cum

inst_mtbo, cum_mtbo = compute_regrets(res_mtbo)
inst_optuna, cum_optuna = compute_regrets(res_optuna)

dist_mtbo = float(np.mean(np.linalg.norm(res_mtbo.state_history - u_star, axis=-1)))
dist_optuna = float(np.mean(np.linalg.norm(res_optuna.state_history - u_star, axis=-1)))
final_dist_mtbo = float(np.linalg.norm(res_mtbo.state_history[-1] - u_star))
final_dist_optuna = float(np.linalg.norm(res_optuna.state_history[-1] - u_star))

print("\n--- Final Benchmark Summary (512 Evaluations) ---", flush=True)
print(f"MTBO:   Total Regret = {cum_mtbo[-1]:.2f} (log10={np.log10(cum_mtbo[-1]):.4f}), Mean State Dist = {dist_mtbo:.4f}, Final Dist = {final_dist_mtbo:.4f}", flush=True)
print(f"Optuna: Total Regret = {cum_optuna[-1]:.2f} (log10={np.log10(cum_optuna[-1]):.4f}), Mean State Dist = {dist_optuna:.4f}, Final Dist = {final_dist_optuna:.4f}", flush=True)

# Save comparison summary CSV
summary_csv = os.path.join(bench_dir, "benchmark_comparison_summary.csv")
with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Optimizer", "Total Regret", "log10(Regret)", "Mean State Dist", "Final State Dist", "Wall Time (s)"])
    writer.writerow(["Refactored MTBO", cum_mtbo[-1], np.log10(cum_mtbo[-1]), dist_mtbo, final_dist_mtbo, t_mtbo])
    writer.writerow(["Optuna (TPE)", cum_optuna[-1], np.log10(cum_optuna[-1]), dist_optuna, final_dist_optuna, t_optuna])

# 6. Plots
steps = np.arange(len(cum_mtbo))
t_axis = steps * 0.001

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(t_axis, inst_mtbo, label=f"MTBO (Final={cum_mtbo[-1]:.0f})", color="tab:blue", alpha=0.8)
plt.plot(t_axis, inst_optuna, label=f"Optuna TPE (Final={cum_optuna[-1]:.0f})", color="tab:orange", alpha=0.8)
plt.yscale("log")
plt.xlabel("Time (s)")
plt.ylabel("Instantaneous Regret $R_1(t) + R_2(t)$")
plt.title("Instantaneous Regret Dynamics (512 Evals)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(t_axis, cum_mtbo, label=f"MTBO (Sum={cum_mtbo[-1]:.0f})", color="tab:blue", lw=2)
plt.plot(t_axis, cum_optuna, label=f"Optuna TPE (Sum={cum_optuna[-1]:.0f})", color="tab:orange", lw=2)
plt.yscale("log")
plt.xlabel("Time (s)")
plt.ylabel("Cumulative Regret")
plt.title("Cumulative Regret Comparison (512 Evals)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(bench_dir, "cumulative_regret_comparison.png"), dpi=300)
plt.close()

# 12 Subplot comparison figure
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes[0, 0].plot(t_axis, res_mtbo.state_history[:, 0], label=r"State $\hat{u}_1$", color="navy")
axes[0, 0].axhline(u_star[0], color="red", linestyle="--", label=r"NE $u_1^*$")
axes[0, 0].set_ylim([0, 50])
axes[0, 0].set_title("MTBO Player 1 State")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(t_axis, res_mtbo.state_history[:, 1], label=r"State $\hat{u}_2$", color="darkgreen")
axes[0, 1].axhline(u_star[1], color="red", linestyle="--", label=r"NE $u_2^*$")
axes[0, 1].set_ylim([0, 50])
axes[0, 1].set_title("MTBO Player 2 State")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[0, 2].plot(t_axis, res_mtbo.action_history[:, 0], label=r"Action $u_1$", color="blue", alpha=0.5)
axes[0, 2].plot(t_axis, res_mtbo.action_history[:, 1], label=r"Action $u_2$", color="green", alpha=0.5)
axes[0, 2].axhline(u_star[0], color="red", linestyle="--")
axes[0, 2].axhline(u_star[1], color="red", linestyle="--")
axes[0, 2].set_ylim([0, 50])
axes[0, 2].set_title("MTBO Actions $(u_1, u_2)$")
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend()

axes[1, 0].plot(t_axis, res_mtbo.payoff_history[:, 0], label=r"Payoff $J_1$", color="navy")
axes[1, 0].axhline(opt_payoffs[0], color="red", linestyle="--", label="NE Payoff")
axes[1, 0].set_ylim([-1000, 1000])
axes[1, 0].set_title("MTBO Player 1 Payoff")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(t_axis, res_mtbo.payoff_history[:, 1], label=r"Payoff $J_2$", color="darkgreen")
axes[1, 1].axhline(opt_payoffs[1], color="red", linestyle="--", label="NE Payoff")
axes[1, 1].set_ylim([-1000, 1000])
axes[1, 1].set_title("MTBO Player 2 Payoff")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

axes[1, 2].plot(t_axis, inst_mtbo, label="Instant Regret", color="purple")
axes[1, 2].set_yscale("log")
axes[1, 2].set_title("MTBO Regret Dynamics")
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend()

axes[2, 0].plot(t_axis, res_optuna.state_history[:, 0], label=r"State $\hat{u}_1$", color="navy")
axes[2, 0].axhline(u_star[0], color="red", linestyle="--", label=r"NE $u_1^*$")
axes[2, 0].set_ylim([0, 50])
axes[2, 0].set_title("Optuna TPE Player 1 State")
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].legend()

axes[2, 1].plot(t_axis, res_optuna.state_history[:, 1], label=r"State $\hat{u}_2$", color="darkgreen")
axes[2, 1].axhline(u_star[1], color="red", linestyle="--", label=r"NE $u_2^*$")
axes[2, 1].set_ylim([0, 50])
axes[2, 1].set_title("Optuna TPE Player 2 State")
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].legend()

axes[2, 2].plot(t_axis, res_optuna.action_history[:, 0], label=r"Action $u_1$", color="blue", alpha=0.5)
axes[2, 2].plot(t_axis, res_optuna.action_history[:, 1], label=r"Action $u_2$", color="green", alpha=0.5)
axes[2, 2].axhline(u_star[0], color="red", linestyle="--")
axes[2, 2].axhline(u_star[1], color="red", linestyle="--")
axes[2, 2].set_ylim([0, 50])
axes[2, 2].set_title("Optuna TPE Actions $(u_1, u_2)$")
axes[2, 2].grid(True, alpha=0.3)
axes[2, 2].legend()

axes[3, 0].plot(t_axis, res_optuna.payoff_history[:, 0], label=r"Payoff $J_1$", color="navy")
axes[3, 0].axhline(opt_payoffs[0], color="red", linestyle="--", label="NE Payoff")
axes[3, 0].set_ylim([-1000, 1000])
axes[3, 0].set_title("Optuna TPE Player 1 Payoff")
axes[3, 0].grid(True, alpha=0.3)
axes[3, 0].legend()

axes[3, 1].plot(t_axis, res_optuna.payoff_history[:, 1], label=r"Payoff $J_2$", color="darkgreen")
axes[3, 1].axhline(opt_payoffs[1], color="red", linestyle="--", label="NE Payoff")
axes[3, 1].set_ylim([-1000, 1000])
axes[3, 1].set_title("Optuna TPE Player 2 Payoff")
axes[3, 1].grid(True, alpha=0.3)
axes[3, 1].legend()

axes[3, 2].plot(t_axis, inst_optuna, label="Instant Regret", color="purple")
axes[3, 2].set_yscale("log")
axes[3, 2].set_title("Optuna TPE Regret Dynamics")
axes[3, 2].grid(True, alpha=0.3)
axes[3, 2].legend()

plt.tight_layout()
plt.savefig(os.path.join(bench_dir, "trajectory_comparison_12subplots.png"), dpi=300)
plt.close()

print(f"\nAll benchmark artifacts saved to {bench_dir}/ successfully!", flush=True)


## 1. 100-Game Synthetic Ensemble Generation & Diverse Representative Game Selection

We sample an ensemble of 100 strongly monotone quadratic games ($N=2, m=1$) with guaranteed interior Nash equilibria.
The instantaneous regret at step $t$ for player $i$ against opponent actions $u_{-i}(t)$ is:
$$u_i^{\text{unc}}(u_{-i}) = -H_{ii}^{-1} (H_{ij} u_j + d_i), \qquad u_i^*(u_{-i}) = \text{clip}(u_i^{\text{unc}}(u_{-i}), \text{lb}_i, \text{ub}_i)$$
$$R_i(t) = J_i(u_i^*(u_{-i}(t)), u_{-i}(t)) - J_i(u_i(t), u_{-i}(t)) \ge 0$$
MTBO minimizes the logarithmic total systemic regret over 20,000 steps ($T = 20,000$):
$$\text{Cost}(G, \text{history}) = \log_{10}\left(\max\left(\sum_{t=0}^T \sum_{i=1}^N R_i(t), \; 10^{-12}\right)\right)$$

Five diverse representative games are selected programmatically across the entire 100-game ensemble:
1. **Extreme Ill-Conditioned:** $\max \kappa(H_{\text{sym}}) = \lambda_{\max} / \lambda_{\min}$
2. **Well-Behaved / Isotropic:** $\min \kappa(H_{\text{sym}}) \approx 1.0$
3. **Narrow Action Bounds:** $\min \prod (\text{ub}_i - \text{lb}_i)$
4. **Highly Asymmetric Coupling:** $\max \|H - H^T\|_F$
5. **Boundary-Adjacent Equilibrium:** $\min \text{dist}(u^*, \partial \Omega)$


In [1]:
import os
import time
import csv
import numpy as np
import matplotlib.pyplot as plt
import optuna
from scipy import stats

from payoffnash import (
    Game,
    QuadraticGame,
    generate_quadratic_games,
    MetaParamOptimization,
    ZOGrandientPlay,
    NESeeking,
    NESeeking5D,
    EstimateHistory,
    quadratic_game_regret_cost,
)

# 1. Output directory for 100-game ensemble artifacts
data_dir = "data/06_saa_tuned_100_games"
os.makedirs(data_dir, exist_ok=True)

# 2. Sample 100 strongly monotone quadratic games with N=2, m=1 and NE strictly in the interior
games = generate_quadratic_games(
    ensemble_size=100,
    N=2,
    m=1,
    alpha=0.05,
    enforce_interior=True,
    seed=14062003,
)

def compute_condition_number(game: QuadraticGame) -> float:
    H_sym = 0.5 * (game.H + game.H.T)
    w = np.linalg.eigvalsh(-H_sym)
    return float(w[-1] / max(w[0], 1e-12))

def compute_action_volume(game: QuadraticGame) -> float:
    vol = 1.0
    for s in game.action_sets:
        for iv in s:
            vol *= (iv[1] - iv[0])
    return float(vol)

def compute_skewness(game: QuadraticGame) -> float:
    return float(np.linalg.norm(game.H - game.H.T, ord="fro"))

def compute_boundary_distance(game: QuadraticGame) -> float:
    lb = np.array([[iv[0] for iv in s] for s in game.action_sets]).ravel()
    ub = np.array([[iv[1] for iv in s] for s in game.action_sets]).ravel()
    u_star = game.nash_equilibrium.ravel()
    return float(np.min(np.minimum(u_star - lb, ub - u_star)))

def select_representative_games(games: list[QuadraticGame]) -> list[tuple[int, str, QuadraticGame]]:
    cond_nums = [compute_condition_number(g) for g in games]
    vols = [compute_action_volume(g) for g in games]
    skews = [compute_skewness(g) for g in games]
    b_dists = [compute_boundary_distance(g) for g in games]

    selected_indices: set[int] = set()
    representative_games: list[tuple[int, str, QuadraticGame]] = []

    # 1. Ill-Conditioned (max kappa)
    idx_ill = int(np.argmax(cond_nums))
    selected_indices.add(idx_ill)
    representative_games.append((idx_ill, f"Extreme Ill-Conditioned (kappa={cond_nums[idx_ill]:.2f})", games[idx_ill]))

    # 2. Isotropic (min kappa)
    for idx in np.argsort(cond_nums):
        if idx not in selected_indices:
            selected_indices.add(int(idx))
            representative_games.append((int(idx), f"Well-Behaved / Isotropic (kappa={cond_nums[idx]:.2f})", games[idx]))
            break

    # 3. Narrow Bounds (min volume)
    for idx in np.argsort(vols):
        if idx not in selected_indices:
            selected_indices.add(int(idx))
            representative_games.append((int(idx), f"Narrow Action Bounds (vol={vols[idx]:.4f})", games[idx]))
            break

    # 4. Asymmetric Coupling (max skewness)
    for idx in np.argsort(skews)[::-1]:
        if idx not in selected_indices:
            selected_indices.add(int(idx))
            representative_games.append((int(idx), f"Highly Asymmetric Coupling (skew={skews[idx]:.3f})", games[idx]))
            break

    # 5. Boundary-Adjacent Equilibrium (min distance to boundary)
    for idx in np.argsort(b_dists):
        if idx not in selected_indices:
            selected_indices.add(int(idx))
            representative_games.append((int(idx), f"Boundary-Adjacent Equilibrium (dist={b_dists[idx]:.3f})", games[idx]))
            break

    return representative_games

rep_games = select_representative_games(games)
print(f"Generated {len(games)} strongly monotone quadratic games. Selected 5 representative games across the entire ensemble:")
for g_idx, category, g in rep_games:
    u_star = g.nash_equilibrium.ravel()
    print(f"  Game {g_idx:2d} | Category: {category:45s} | True NE: [{u_star[0]:.4f}, {u_star[1]:.4f}]")

# 3. Generalized Logarithmic Total Regret Cost Function
def quadratic_game_regret_cost(game: QuadraticGame, history: EstimateHistory) -> float:
    actions = history.action_history
    H = game.H
    d = game.d
    lb = np.array([[iv[0] for iv in s] for s in game.action_sets]).ravel()
    ub = np.array([[iv[1] for iv in s] for s in game.action_sets]).ravel()

    u0 = actions[:, 0]
    u1 = actions[:, 1]

    # Constrained best responses
    u0_unc = -(H[0, 1] * u1 + d[0]) / H[0, 0]
    u0_br = np.clip(u0_unc, lb[0], ub[0])

    u1_unc = -(H[1, 0] * u0 + d[1]) / H[1, 1]
    u1_br = np.clip(u1_unc, lb[1], ub[1])

    # Payoff differences J_i(u_i^*, u_{-i}) - J_i(u_i, u_{-i})
    diff0 = (
        0.5 * H[0, 0] * (u0_br**2 - u0**2)
        + H[0, 1] * (u0_br - u0) * u1
        + d[0] * (u0_br - u0)
    )
    regret0 = np.maximum(diff0, 0.0)

    diff1 = (
        0.5 * H[1, 1] * (u1_br**2 - u1**2)
        + H[1, 0] * u0 * (u1_br - u1)
        + d[1] * (u1_br - u1)
    )
    regret1 = np.maximum(diff1, 0.0)

    total_regret = float(np.sum(regret0 + regret1))
    return float(np.log10(max(total_regret, 1e-12)))

/home/bruno/Documents/PayoffBasedNEComparison/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generated 100 strongly monotone quadratic games. Selected 5 representative games across the entire ensemble:
  Game 11 | Category: Extreme Ill-Conditioned (kappa=15.16)         | True NE: [0.2890, 0.0735]
  Game 70 | Category: Well-Behaved / Isotropic (kappa=1.05)         | True NE: [0.3248, 0.4164]
  Game 97 | Category: Narrow Action Bounds (vol=0.0818)             | True NE: [-0.0507, 0.0480]
  Game 40 | Category: Highly Asymmetric Coupling (skew=2.781)       | True NE: [0.1650, 0.2749]
  Game 17 | Category: Boundary-Adjacent Equilibrium (dist=0.006)    | True NE: [0.1908, 0.6364]


## 2. Zero-Order Gradient Play: Batched MTBO ($4 \times 25$ Games)

Zero-Order Gradient Play ([`ZOGrandientPlay`](file:///home/bruno/Documents/PayoffBasedNEComparison/payoffnash/zo_gradient_play.py)) is tuned across 4 sequential batches of 25 games each:
- **Search Space (2 Continuous Dimensions):** $v \in [0.01, 100.0]$ (gradient step size), $a \in [0.005, 2.0]$ (dimensionless perturbation ratio relative to bottleneck bound width).
- **Scale Invariance (`scaled_params`):** `scaled_params = {"a": "min_box_width"}` scales the effective perturbation radius $a_{\text{eff}} = a \cdot \min_i (ub_i - lb_i)$ across games with varying boundary sizes.
- **Fixed Parameters:** `epsilon = 1e-4` (decay stabilization parameter), `estimation_type = "one-point"`.
- **Batch Budget:** $n_{\text{init}} = 16$ (Sobol quasi-random block, $16 \times 25 = 400$ training points per batch), $n_{\text{bo}} \le 200$ with stricter stagnation stopping (`patience = 10`, `eps_ei = 1e-5`, `tol_stagnation = 1e-5`).
- **Option B Polish:** Continuous surrogate L-BFGS-B polish across task posteriors followed by parallel verification.


In [ ]:
# 2. ZOGrandientPlay MTBO (4 x 25 Games Batched MTBO, Total Regret Minimization)
zo_search_space = {
    "v": (0.01, 100.0),
    "a": (0.005, 2.0),
}
zo_fixed_params = {
    "epsilon": 1e-4,
}
zo_scaled_params = {
    "a": "min_box_width",
}

batch_size = 25
n_batches = 4
zo_batch_opts: list[MetaParamOptimization] = []
zo_best_thetas: list[np.ndarray] = [None] * len(games)
zo_candidate_defaults: list[np.ndarray] = []
zo_batch_val_metrics: list[dict] = []

print("=== Starting ZOGrandientPlay Batched MTBO (4 batches of 25 games, Multi-Core) ===", flush=True)
t0_zo = time.time()

for b in range(n_batches):
    b_start = b * batch_size
    b_end = (b + 1) * batch_size
    batch_games = games[b_start:b_end]
    print(f"\n--- Running ZOGrandientPlay MTBO Batch {b + 1}/{n_batches} (Games {b_start} to {b_end - 1}) ---", flush=True)

    zo_opt_b = MetaParamOptimization(
        game_ensemble=batch_games,
        search_space=zo_search_space,
        algorithm_class=ZOGrandientPlay,
        cost_function=quadratic_game_regret_cost,
        output_path=os.path.join(data_dir, f"zo_batch_{b}_results.csv"),
        num_steps=20000,
        n_init=64,
        max_bo_steps=200,
        min_bo_steps=20,
        sim_per_param=5,
        eps_ei=1e-5,
        patience=20,
        tol_stagnation=1e-5,
        seed=14062003 + b,
        fixed_params=zo_fixed_params,
        scaled_params=zo_scaled_params,
    )

    zo_results_b = zo_opt_b.run_optimization_loop()
    zo_val_b = zo_opt_b.validate()
    zo_batch_opts.append(zo_opt_b)
    zo_batch_val_metrics.append(zo_val_b)

    for local_idx in range(batch_size):
        zo_best_thetas[b_start + local_idx] = zo_opt_b.best_thetas[local_idx]
    zo_candidate_defaults.append(zo_opt_b.best_theta_def)

    print(f"Batch {b + 1} finished in {zo_opt_b.n_bo_steps_evaluated} BO steps. Stop reason: {zo_opt_b.stop_reason}")

t_zo = time.time() - t0_zo
zo_total_bo_steps = sum(opt.n_bo_steps_evaluated for opt in zo_batch_opts)
print(f"\nAll 4 ZOGrandientPlay batches complete in {t_zo:.2f}s ({t_zo/60:.2f} min). Total BO steps: {zo_total_bo_steps}")

# Aggregate surrogate validation metrics across batches
zo_val_metrics = {
    "loo_rmse": float(np.mean([m["loo_rmse"] for m in zo_batch_val_metrics])),
    "loo_smse": float(np.mean([m["loo_smse"] for m in zo_batch_val_metrics])),
    "calibration_std": float(np.mean([m["calibration_std"] for m in zo_batch_val_metrics])),
    "anomalies_detected": int(sum(m["anomalies_detected"] for m in zo_batch_val_metrics)),
    "refinements_executed": int(sum(m["refinements_executed"] for m in zo_batch_val_metrics)),
}
print("Aggregated ZOGrandientPlay Validation Metrics:", zo_val_metrics)

# Option B: Global Consensus Default Determination
print("\n--- Determining Global Consensus Default (Option B) across All 100 Games ---", flush=True)
zo_mean_default = np.mean(np.array(zo_candidate_defaults), axis=0)
zo_candidates = zo_candidate_defaults + [zo_mean_default]

zo_evaluator = MetaParamOptimization(
    game_ensemble=games,
    search_space=zo_search_space,
    algorithm_class=ZOGrandientPlay,
    cost_function=quadratic_game_regret_cost,
    num_steps=20000,
    fixed_params=zo_fixed_params,
    scaled_params=zo_scaled_params,
)

zo_eval_requests = [(cand, g_idx) for cand in zo_candidates for g_idx in range(len(games))]
zo_eval_losses = np.array(zo_evaluator.evaluate_batch(zo_eval_requests)).reshape(len(zo_candidates), len(games))
zo_cand_means = np.mean(zo_eval_losses, axis=1)

for c_i, c_mean in enumerate(zo_cand_means):
    c_label = f"Batch {c_i + 1} default" if c_i < n_batches else "Mean of batch defaults"
    print(f"  Candidate {c_i} ({c_label}): Mean Regret Loss = {c_mean:.4f}")

zo_best_cand_idx = int(np.argmin(zo_cand_means))
zo_best_theta_def = zo_candidates[zo_best_cand_idx]
zo_losses_def = zo_eval_losses[zo_best_cand_idx]
print(f"Selected Global Consensus Default: Candidate {zo_best_cand_idx} with Mean Loss = {zo_cand_means[zo_best_cand_idx]:.4f}")
print("Global Consensus Default Parameters:", {k: zo_best_theta_def[i] for i, k in enumerate(zo_evaluator.param_names)})

# Evaluate tailored optima on all 100 games in parallel
print("Evaluating all 100 tailored optima in parallel...", flush=True)
zo_tailored_requests = [(zo_best_thetas[j], j) for j in range(len(games))]
zo_losses_opt = np.array(zo_evaluator.evaluate_batch(zo_tailored_requests))
zo_tunability = zo_losses_def - zo_losses_opt
print(f"ZOGP Mean Loss Tailored: {np.mean(zo_losses_opt):.4f} | Default: {np.mean(zo_losses_def):.4f} | Mean Tunability Gain: {np.mean(zo_tunability):.4f}")

import gc
gc.collect()


=== Starting ZOGrandientPlay Batched MTBO (4 batches of 25 games, Multi-Core) ===

--- Running ZOGrandientPlay MTBO Batch 1/4 (Games 0 to 24) ---


## 3. Zero-Order Gradient Play: Optuna Baseline & Trajectory Visualizations

We run a dedicated 100-trial single-game Optuna TPE benchmark on each of the 5 representative games over 20,000 steps using identical scale-invariant parameterization ($v \in [0.01, 100.0]$, $a \in [0.005, 2.0]$ scaled by $\min_i (ub_i - lb_i)$).
For each representative game, we plot a 2x2 subplot comparing action trajectories:
- **Top-Left / Bottom-Left:** Player 1 and Player 2 actions comparing MTBO Tailored Optima vs. Optuna TPE vs. True NE.
- **Top-Right / Bottom-Right:** Player 1 and Player 2 actions comparing MTBO Global Consensus Default vs. True NE.


In [ ]:
# 3. ZOGrandientPlay: Optuna Per-Game Baseline & 3-Way Trajectory Visualizations on 5 Representative Games
num_sim_steps = 20000
optuna.logging.set_verbosity(optuna.logging.WARNING)

def run_optuna_zo_game(game: QuadraticGame, seed: int = 14062003) -> tuple[dict[str, float], float]:
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in game.action_sets], dtype=float)
    min_w = float(np.min(bounds))

    def objective(trial: optuna.Trial) -> float:
        v = trial.suggest_float("v", 0.01, 100.0, log=True)
        alpha = trial.suggest_float("a", 0.005, 2.0, log=True)
        est = ZOGrandientPlay(
            game,
            v=v,
            a=alpha * min_w,
            epsilon=1e-4,
        )
        res = est.estimate(np.zeros((game.N, 1)), num_steps=num_sim_steps)
        return quadratic_game_regret_cost(game, res)

    sampler = optuna.samplers.TPESampler(n_startup_trials=20, seed=seed)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=100)
    return study.best_params, study.best_value

zo_optuna_results: dict[int, tuple[dict[str, float], float, float]] = {}
t0_zo_optuna_5 = time.time()

for g_idx, category, g in rep_games:
    print(f"Running Optuna on Game {g_idx} ({category})...", flush=True)
    t0_g = time.time()
    best_params, best_val = run_optuna_zo_game(g, seed=14062003 + g_idx)
    t_g = time.time() - t0_g
    zo_optuna_results[g_idx] = (best_params, best_val, t_g)

    # 1. MTBO Tailored
    est_opt = zo_evaluator.build_estimator(zo_best_thetas[g_idx], g)
    res_opt = est_opt.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_opt = quadratic_game_regret_cost(g, res_opt)

    # 2. MTBO Consensus Default
    est_def = zo_evaluator.build_estimator(zo_best_theta_def, g)
    res_def = est_def.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_def = quadratic_game_regret_cost(g, res_def)

    # 3. Optuna Direct
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in g.action_sets], dtype=float)
    min_w = float(np.min(bounds))
    est_tpe = ZOGrandientPlay(
        g,
        v=best_params["v"],
        a=best_params["a"] * min_w,
        epsilon=1e-4,
    )
    res_tpe = est_tpe.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_tpe = quadratic_game_regret_cost(g, res_tpe)

    x_star = g.nash_equilibrium.ravel()
    t_steps = np.arange(len(res_opt.action_history))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    title_str = (
        f"ZOGrandientPlay — Game {g_idx}: {category}\n"
        f"Regret Loss: Optuna={cost_tpe:.4f} | MTBO Tailored={cost_opt:.4f} | MTBO Consensus Default={cost_def:.4f} | NE: [{x_star[0]:.3f}, {x_star[1]:.3f}]"
    )
    fig.suptitle(title_str, fontsize=12, fontweight="bold")

    # Player 1 Actions
    axes[0, 0].plot(t_steps, res_opt.action_history[:, 0], color="tab:blue", lw=1.2, label=f"MTBO Tailored (Loss: {cost_opt:.3f})")
    axes[0, 0].plot(t_steps, res_tpe.action_history[:, 0], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna (Loss: {cost_tpe:.3f})")
    axes[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label=f"True u1* ({x_star[0]:.3f})")
    axes[0, 0].set_title("Player 1: Tailored Optima Comparison")
    axes[0, 0].set_ylabel("Action u1")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(loc="best")

    axes[0, 1].plot(t_steps, res_def.action_history[:, 0], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default (Loss: {cost_def:.3f})")
    axes[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label=f"True u1* ({x_star[0]:.3f})")
    axes[0, 1].set_title("Player 1: Universal Consensus Default")
    axes[0, 1].set_ylabel("Action u1")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(loc="best")

    # Player 2 Actions
    axes[1, 0].plot(t_steps, res_opt.action_history[:, 1], color="tab:blue", lw=1.2, label=f"MTBO Tailored (Loss: {cost_opt:.3f})")
    axes[1, 0].plot(t_steps, res_tpe.action_history[:, 1], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna (Loss: {cost_tpe:.3f})")
    axes[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label=f"True u2* ({x_star[1]:.3f})")
    axes[1, 0].set_title("Player 2: Tailored Optima Comparison")
    axes[1, 0].set_xlabel("Iteration Step")
    axes[1, 0].set_ylabel("Action u2")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(loc="best")

    axes[1, 1].plot(t_steps, res_def.action_history[:, 1], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default (Loss: {cost_def:.3f})")
    axes[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label=f"True u2* ({x_star[1]:.3f})")
    axes[1, 1].set_title("Player 2: Universal Consensus Default")
    axes[1, 1].set_xlabel("Iteration Step")
    axes[1, 1].set_ylabel("Action u2")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir, f"zo_rep_game_{g_idx}.png"), dpi=300)
    plt.show()
    plt.close(fig)
    plt.close('all')
    import gc
    gc.collect()

t_zo_optuna_5 = time.time() - t0_zo_optuna_5
print(f"All 5 Optuna runs for ZOGP completed in {t_zo_optuna_5:.2f}s ({t_zo_optuna_5/60:.2f} min).")


## 4. Continuous-Time Extremum Seeking (NESeeking5D): Batched MTBO ($4 \times 25$ Games)

Continuous-Time Extremum Seeking ([`NESeeking5D`](file:///home/bruno/Documents/PayoffBasedNEComparison/payoffnash/ne_seeking.py)) is tuned across 4 sequential batches of 25 games each using the 5-dimensional search space derived from our sensitivity analysis (`04_neseeking_dimension_reduction_study.ipynb`):
- **Search Space (5 Continuous Dimensions):**
  - Integration gains: $k[0,0], k[1,0] \in [0.01, 100.0]$
  - Perturbation amplitudes: $a[0,0], a[1,0] \in [0.005, 2.0]$ (dimensionless perturbation ratio relative to each player's action box width)
  - Base dither frequency: $\omega_1 \in [1.0, 50.0]$
- **Scale Invariance (`scaled_params`):** `scaled_params = {"a[0,0]": "player_box_width", "a[1,0]": "player_box_width"}` scales the perturbation amplitudes $a_i = \alpha_i \cdot (ub_i - lb_i)$ per player.
- **Coupled Degrees of Freedom (Option A):** $\omega_2 = 1.35 \cdot \omega_1$, $\omega_{h1} = 0.20 \cdot \omega_1$, $\omega_{h2} = 0.20 \cdot \omega_2$.
- **Fixed Parameters:** `dt = 0.001` (forward Euler timestep).
- **Batch Budget:** $n_{\text{init}} = 32$ (Sobol quasi-random block, $32 \times 25 = 800$ training points per batch), $n_{\text{bo}} \le 200$ with stricter stagnation stopping (`patience = 10`, `eps_ei = 1e-5`, `tol_stagnation = 1e-5`).
- **Option B Polish:** Continuous surrogate L-BFGS-B polish across task posteriors followed by parallel verification.


In [ ]:
# 4. NESeeking5D MTBO (4 x 25 Games Batched MTBO, 5D Search Space, Total Regret Minimization)
ne_search_space = {
    "k[0,0]": (0.01, 100.0),
    "k[1,0]": (0.01, 100.0),
    "a[0,0]": (0.005, 2.0),
    "a[1,0]": (0.005, 2.0),
    "omega1": (1.0, 50.0),
}
ne_fixed_params = {
    "dt": 0.001,
}
ne_scaled_params = {
    "a[0,0]": "player_box_width",
    "a[1,0]": "player_box_width",
}

batch_size = 25
n_batches = 4
ne_batch_opts: list[MetaParamOptimization] = []
ne_best_thetas: list[np.ndarray] = [None] * len(games)
ne_candidate_defaults: list[np.ndarray] = []
ne_batch_val_metrics: list[dict] = []

print("=== Starting NESeeking5D Batched MTBO (4 batches of 25 games, Multi-Core) ===", flush=True)
t0_ne = time.time()

for b in range(n_batches):
    b_start = b * batch_size
    b_end = (b + 1) * batch_size
    batch_games = games[b_start:b_end]
    print(f"\n--- Running NESeeking5D MTBO Batch {b + 1}/{n_batches} (Games {b_start} to {b_end - 1}) ---", flush=True)

    ne_opt_b = MetaParamOptimization(
        game_ensemble=batch_games,
        search_space=ne_search_space,
        algorithm_class=NESeeking5D,
        cost_function=quadratic_game_regret_cost,
        output_path=os.path.join(data_dir, f"ne_batch_{b}_results.csv"),
        num_steps=20000,
        n_init=64,
        max_bo_steps=200,
        min_bo_steps=20,
        sim_per_param=1,
        eps_ei=1e-5,
        patience=20,
        tol_stagnation=1e-5,
        seed=14062003 + b,
        fixed_params=ne_fixed_params,
        scaled_params=ne_scaled_params,
    )

    ne_results_b = ne_opt_b.run_optimization_loop()
    ne_val_b = ne_opt_b.validate()
    ne_batch_opts.append(ne_opt_b)
    ne_batch_val_metrics.append(ne_val_b)

    for local_idx in range(batch_size):
        ne_best_thetas[b_start + local_idx] = ne_opt_b.best_thetas[local_idx]
    ne_candidate_defaults.append(ne_opt_b.best_theta_def)

    print(f"Batch {b + 1} finished in {ne_opt_b.n_bo_steps_evaluated} BO steps. Stop reason: {ne_opt_b.stop_reason}")

t_ne = time.time() - t0_ne
ne_total_bo_steps = sum(opt.n_bo_steps_evaluated for opt in ne_batch_opts)
print(f"\nAll 4 NESeeking5D batches complete in {t_ne:.2f}s ({t_ne/60:.2f} min). Total BO steps: {ne_total_bo_steps}")

# Aggregate surrogate validation metrics across batches
ne_val_metrics = {
    "loo_rmse": float(np.mean([m["loo_rmse"] for m in ne_batch_val_metrics])),
    "loo_smse": float(np.mean([m["loo_smse"] for m in ne_batch_val_metrics])),
    "calibration_std": float(np.mean([m["calibration_std"] for m in ne_batch_val_metrics])),
    "anomalies_detected": int(sum(m["anomalies_detected"] for m in ne_batch_val_metrics)),
    "refinements_executed": int(sum(m["refinements_executed"] for m in ne_batch_val_metrics)),
}
print("Aggregated NESeeking5D Validation Metrics:", ne_val_metrics)

# Option B: Global Consensus Default Determination
print("\n--- Determining Global Consensus Default (Option B) across All 100 Games ---", flush=True)
ne_mean_default = np.mean(np.array(ne_candidate_defaults), axis=0)
ne_candidates = ne_candidate_defaults + [ne_mean_default]

ne_evaluator = MetaParamOptimization(
    game_ensemble=games,
    search_space=ne_search_space,
    algorithm_class=NESeeking5D,
    cost_function=quadratic_game_regret_cost,
    num_steps=20000,
    fixed_params=ne_fixed_params,
    scaled_params=ne_scaled_params,
)

ne_eval_requests = [(cand, g_idx) for cand in ne_candidates for g_idx in range(len(games))]
ne_eval_losses = np.array(ne_evaluator.evaluate_batch(ne_eval_requests)).reshape(len(ne_candidates), len(games))
ne_cand_means = np.mean(ne_eval_losses, axis=1)

for c_i, c_mean in enumerate(ne_cand_means):
    c_label = f"Batch {c_i + 1} default" if c_i < n_batches else "Mean of batch defaults"
    print(f"  Candidate {c_i} ({c_label}): Mean Regret Loss = {c_mean:.4f}")

ne_best_cand_idx = int(np.argmin(ne_cand_means))
ne_best_theta_def = ne_candidates[ne_best_cand_idx]
ne_losses_def = ne_eval_losses[ne_best_cand_idx]
print(f"Selected Global Consensus Default: Candidate {ne_best_cand_idx} with Mean Loss = {ne_cand_means[ne_best_cand_idx]:.4f}")
print("Global Consensus Default Parameters:", {k: ne_best_theta_def[i] for i, k in enumerate(ne_evaluator.param_names)})

# Evaluate tailored optima on all 100 games in parallel
print("Evaluating all 100 tailored optima in parallel...", flush=True)
ne_tailored_requests = [(ne_best_thetas[j], j) for j in range(len(games))]
ne_losses_opt = np.array(ne_evaluator.evaluate_batch(ne_tailored_requests))
ne_tunability = ne_losses_def - ne_losses_opt
print(f"NES Mean Loss Tailored: {np.mean(ne_losses_opt):.4f} | Default: {np.mean(ne_losses_def):.4f} | Mean Tunability Gain: {np.mean(ne_tunability):.4f}")

import gc
gc.collect()


## 5. Continuous-Time Extremum Seeking (NESeeking5D): Optuna Baseline & Trajectory Visualizations

We run a dedicated 100-trial single-game Optuna TPE benchmark on each of the 5 representative games over 20,000 steps using `NESeeking5D` with identical scale-invariant parameterization ($a[0,0], a[1,0] \in [0.005, 2.0]$ scaled by each player's action box width).
For each representative game, we plot a 2x2 subplot comparing action trajectories:
- **Top-Left / Bottom-Left:** Player 1 and Player 2 actions comparing MTBO Tailored Optima vs. Optuna TPE vs. True NE.
- **Top-Right / Bottom-Right:** Player 1 and Player 2 actions comparing MTBO Global Consensus Default vs. True NE.


In [ ]:
# 5. NESeeking5D: Optuna Per-Game Baseline & 3-Way Trajectory Visualizations on 5 Representative Games
def run_optuna_ne_game(game: QuadraticGame, seed: int = 14062003) -> tuple[dict[str, float], float]:
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in game.action_sets], dtype=float)
    w0, w1 = float(bounds[0, 0]), float(bounds[1, 0])

    def objective(trial: optuna.Trial) -> float:
        k0 = trial.suggest_float("k[0,0]", 0.01, 100.0, log=True)
        k1 = trial.suggest_float("k[1,0]", 0.01, 100.0, log=True)
        alpha0 = trial.suggest_float("a[0,0]", 0.005, 2.0, log=True)
        alpha1 = trial.suggest_float("a[1,0]", 0.005, 2.0, log=True)
        w1 = trial.suggest_float("omega1", 1.0, 50.0, log=True)

        est = NESeeking5D(
            game,
            k=np.array([[k0], [k1]]),
            a=np.array([[alpha0 * w0], [alpha1 * w1]]),
            omega1=w1,
            dt=0.001,
        )
        res = est.estimate(np.zeros((game.N, 1)), num_steps=num_sim_steps)
        return quadratic_game_regret_cost(game, res)

    sampler = optuna.samplers.TPESampler(n_startup_trials=20, seed=seed)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=100)
    return study.best_params, study.best_value

ne_optuna_results: dict[int, tuple[dict[str, float], float, float]] = {}
t0_ne_optuna_5 = time.time()

for g_idx, category, g in rep_games:
    print(f"Running Optuna on Game {g_idx} ({category})...", flush=True)
    t0_g = time.time()
    best_params, best_val = run_optuna_ne_game(g, seed=14062003 + g_idx)
    t_g = time.time() - t0_g
    ne_optuna_results[g_idx] = (best_params, best_val, t_g)

    # 1. MTBO Tailored
    est_opt = ne_evaluator.build_estimator(ne_best_thetas[g_idx], g)
    res_opt = est_opt.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_opt = quadratic_game_regret_cost(g, res_opt)

    # 2. MTBO Consensus Default
    est_def = ne_evaluator.build_estimator(ne_best_theta_def, g)
    res_def = est_def.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_def = quadratic_game_regret_cost(g, res_def)

    # 3. Optuna Direct
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in g.action_sets], dtype=float)
    w0, w1 = float(bounds[0, 0]), float(bounds[1, 0])
    est_tpe = NESeeking5D(
        g,
        k=np.array([[best_params["k[0,0]"]], [best_params["k[1,0]"]]]),
        a=np.array([[best_params["a[0,0]"] * w0], [best_params["a[1,0]"] * w1]]),
        omega1=best_params["omega1"],
        dt=0.001,
    )
    res_tpe = est_tpe.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_tpe = quadratic_game_regret_cost(g, res_tpe)

    x_star = g.nash_equilibrium.ravel()
    t_steps = np.arange(len(res_opt.action_history)) * 0.001

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    title_str = (
        f"NESeeking5D — Game {g_idx}: {category}\n"
        f"Regret Loss: Optuna={cost_tpe:.4f} | MTBO Tailored={cost_opt:.4f} | MTBO Consensus Default={cost_def:.4f} | NE: [{x_star[0]:.3f}, {x_star[1]:.3f}]"
    )
    fig.suptitle(title_str, fontsize=12, fontweight="bold")

    # Player 1 Actions
    axes[0, 0].plot(t_steps, res_opt.action_history[:, 0], color="tab:blue", lw=1.2, label=f"MTBO Tailored (Loss: {cost_opt:.3f})")
    axes[0, 0].plot(t_steps, res_tpe.action_history[:, 0], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna (Loss: {cost_tpe:.3f})")
    axes[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label=f"True u1* ({x_star[0]:.3f})")
    axes[0, 0].set_title("Player 1: Tailored Optima Comparison")
    axes[0, 0].set_ylabel("Action u1")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(loc="best")

    axes[0, 1].plot(t_steps, res_def.action_history[:, 0], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default (Loss: {cost_def:.3f})")
    axes[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label=f"True u1* ({x_star[0]:.3f})")
    axes[0, 1].set_title("Player 1: Universal Consensus Default")
    axes[0, 1].set_ylabel("Action u1")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(loc="best")

    # Player 2 Actions
    axes[1, 0].plot(t_steps, res_opt.action_history[:, 1], color="tab:blue", lw=1.2, label=f"MTBO Tailored (Loss: {cost_opt:.3f})")
    axes[1, 0].plot(t_steps, res_tpe.action_history[:, 1], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna (Loss: {cost_tpe:.3f})")
    axes[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label=f"True u2* ({x_star[1]:.3f})")
    axes[1, 0].set_title("Player 2: Tailored Optima Comparison")
    axes[1, 0].set_xlabel("Time (s)")
    axes[1, 0].set_ylabel("Action u2")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(loc="best")

    axes[1, 1].plot(t_steps, res_def.action_history[:, 1], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default (Loss: {cost_def:.3f})")
    axes[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label=f"True u2* ({x_star[1]:.3f})")
    axes[1, 1].set_title("Player 2: Universal Consensus Default")
    axes[1, 1].set_xlabel("Time (s)")
    axes[1, 1].set_ylabel("Action u2")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir, f"ne_rep_game_{g_idx}.png"), dpi=300)
    plt.show()
    plt.close(fig)
    plt.close('all')
    import gc
    gc.collect()

t_ne_optuna_5 = time.time() - t0_ne_optuna_5
print(f"All 5 Optuna runs for NES5D completed in {t_ne_optuna_5:.2f}s ({t_ne_optuna_5/60:.2f} min).")


## 6. Quantitative Comparative Evaluation, Statistical Hypothesis Testing & Reporting

We perform comprehensive comparative evaluation across all 100 games:
1. **Representative Games Comparison:** Comparing regret across Optuna, MTBO Tailored, and MTBO Consensus Default.
2. **Statistical Hypothesis Testing:**
   - Paired $t$-test testing $H_0: \mathbb{E}[\Delta \text{Loss}] = 0$
   - Non-parametric Wilcoxon signed-rank test testing $H_0: \text{median}(\Delta \text{Loss}) = 0$ against alternative $H_1: \Delta \text{Loss} > 0$
   - 95% Confidence Intervals for mean tunability gain
3. **Computational Efficiency & Speedup Benchmarking:**
   - Wall-clock MTBO training time vs. extrapolated 100-game Optuna runtime ($T_{\text{Optuna, 100}}^{\text{extrap}} = 20 \times T_{\text{Optuna, 5}}$).
4. **Automated Markdown Report:** Generating `data/05_scaled_100_games_regret/100_games_comparative_report.md`.


In [ ]:
# 6. Quantitative Evaluation, Statistical Testing, Timing Benchmark & Automated Morning Report Generation
import pandas as pd

# 1. Representative Games Comparison Table
rep_rows = []
for g_idx, category, g in rep_games:
    u_star = g.nash_equilibrium.ravel()
    zo_opt_val = float(zo_losses_opt[g_idx])
    zo_def_val = float(zo_losses_def[g_idx])
    zo_tpe_val = float(zo_optuna_results[g_idx][1])
    zo_tun = float(zo_tunability[g_idx])

    ne_opt_val = float(ne_losses_opt[g_idx])
    ne_def_val = float(ne_losses_def[g_idx])
    ne_tpe_val = float(ne_optuna_results[g_idx][1])
    ne_tun = float(ne_tunability[g_idx])

    rep_rows.append({
        "game_id": g_idx,
        "category": category,
        "true_ne": f"[{u_star[0]:.3f}, {u_star[1]:.3f}]",
        "zo_optuna": zo_tpe_val,
        "zo_mtbo_tailored": zo_opt_val,
        "zo_mtbo_default": zo_def_val,
        "zo_tunability": zo_tun,
        "ne_optuna": ne_tpe_val,
        "ne_mtbo_tailored": ne_opt_val,
        "ne_mtbo_default": ne_def_val,
        "ne_tunability": ne_tun,
    })

rep_df = pd.DataFrame(rep_rows)
print("=== Representative Games Comparison Table (Log10 Total Regret) ===")
print(rep_df.to_string(index=False))

rep_summary_path = os.path.join(data_dir, "rep_games_comparison_summary.csv")
rep_df.to_csv(rep_summary_path, index=False)
print(f"\nRepresentative games summary saved to: {rep_summary_path}\n")

# 2. Statistical Hypothesis Testing & Ensemble Statistics across All 100 Games
zo_diff = zo_losses_def - zo_losses_opt
ne_diff = ne_losses_def - ne_losses_opt

zo_t_stat, zo_t_pval = stats.ttest_rel(zo_losses_def, zo_losses_opt)
zo_w_stat, zo_w_pval = stats.wilcoxon(zo_diff, alternative="greater", zero_method="wilcox") if np.any(zo_diff != 0) else (0.0, 1.0)
zo_sem = stats.sem(zo_diff)
zo_ci95 = (np.mean(zo_diff) - 1.96 * zo_sem, np.mean(zo_diff) + 1.96 * zo_sem)

ne_t_stat, ne_t_pval = stats.ttest_rel(ne_losses_def, ne_losses_opt)
ne_w_stat, ne_w_pval = stats.wilcoxon(ne_diff, alternative="greater", zero_method="wilcox") if np.any(ne_diff != 0) else (0.0, 1.0)
ne_sem = stats.sem(ne_diff)
ne_ci95 = (np.mean(ne_diff) - 1.96 * ne_sem, np.mean(ne_diff) + 1.96 * ne_sem)

print("=== Ensemble Statistics across All 100 Games (Log10 Total Regret) ===")
print(f"ZOGP Loss Tailored:   Mean = {np.mean(zo_losses_opt):.4f} +/- {np.std(zo_losses_opt):.4f}, Median = {np.median(zo_losses_opt):.4f}")
print(f"ZOGP Loss Default:    Mean = {np.mean(zo_losses_def):.4f} +/- {np.std(zo_losses_def):.4f}, Median = {np.median(zo_losses_def):.4f}")
print(f"ZOGP Tunability:      Mean = {np.mean(zo_diff):.4f} [95% CI: {zo_ci95[0]:.4f}, {zo_ci95[1]:.4f}], Max = {np.max(zo_diff):.4f}")
print(f"ZOGP Significance:    Paired t p-val = {zo_t_pval:.4e}, Wilcoxon p-val = {zo_w_pval:.4e}\n")

print(f"NES Loss Tailored:    Mean = {np.mean(ne_losses_opt):.4f} +/- {np.std(ne_losses_opt):.4f}, Median = {np.median(ne_losses_opt):.4f}")
print(f"NES Loss Default:     Mean = {np.mean(ne_losses_def):.4f} +/- {np.std(ne_losses_def):.4f}, Median = {np.median(ne_losses_def):.4f}")
print(f"NES Tunability:       Mean = {np.mean(ne_diff):.4f} [95% CI: {ne_ci95[0]:.4f}, {ne_ci95[1]:.4f}], Max = {np.max(ne_diff):.4f}")
print(f"NES Significance:     Paired t p-val = {ne_t_pval:.4e}, Wilcoxon p-val = {ne_w_pval:.4e}\n")

metrics_path = os.path.join(data_dir, "ensemble_100_games_metrics.csv")
with open(metrics_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["game_id", "zo_loss_tailored", "zo_loss_default", "zo_tunability_gain", "ne_loss_tailored", "ne_loss_default", "ne_tunability_gain"])
    for j in range(len(games)):
        writer.writerow([j, zo_losses_opt[j], zo_losses_def[j], zo_tunability[j], ne_losses_opt[j], ne_losses_def[j], ne_tunability[j]])
print(f"All 100 games metrics saved to: {metrics_path}\n")

# 3. Computational Time Benchmarking & 100-Game Optuna Extrapolation
t_zo_optuna_100_extrap = t_zo_optuna_5 * (100.0 / 5.0)
t_ne_optuna_100_extrap = t_ne_optuna_5 * (100.0 / 5.0)

speedup_zo = t_zo_optuna_100_extrap / max(t_zo, 1e-12)
speedup_ne = t_ne_optuna_100_extrap / max(t_ne, 1e-12)

print("=" * 70)
print("COMPUTATIONAL TIME BENCHMARK & 100-GAME OPTUNA EXTRAPOLATION")
print("=" * 70)
print(f"ZOGP MTBO (4x25 Batches):           {t_zo:.2f}s ({t_zo/60:.2f} min) across {zo_total_bo_steps} BO steps")
print(f"ZOGP Optuna (5 Representative):     {t_zo_optuna_5:.2f}s ({t_zo_optuna_5/60:.2f} min)")
print(f"ZOGP Optuna (100 Games Extrapolated): {t_zo_optuna_100_extrap:.2f}s ({t_zo_optuna_100_extrap/60:.2f} min, {t_zo_optuna_100_extrap/3600:.2f} h)")
print(f"ZOGP MTBO Efficiency Speedup:        {speedup_zo:.2f}x\n")

print(f"NES MTBO (4x25 Batches):            {t_ne:.2f}s ({t_ne/60:.2f} min) across {ne_total_bo_steps} BO steps")
print(f"NES Optuna (5 Representative):      {t_ne_optuna_5:.2f}s ({t_ne_optuna_5/60:.2f} min)")
print(f"NES Optuna (100 Games Extrapolated):  {t_ne_optuna_100_extrap:.2f}s ({t_ne_optuna_100_extrap/60:.2f} min, {t_ne_optuna_100_extrap/3600:.2f} h)")
print(f"NES MTBO Efficiency Speedup:         {speedup_ne:.2f}x")
print("=" * 70)

# 4. Generate Comprehensive Morning Markdown Report
# Save Surrogate Validation Metrics CSV
val_csv_path = os.path.join(data_dir, "surrogate_validation_metrics.csv")
val_df = pd.DataFrame([
    {"algorithm": "ZOGrandientPlay", **zo_val_metrics},
    {"algorithm": "NESeeking5D", **ne_val_metrics},
])
val_df.to_csv(val_csv_path, index=False)
print(f"Surrogate validation metrics saved to: {val_csv_path}")

report_path = os.path.join(data_dir, "100_games_comparative_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("# 100-Game Ensemble Benchmark: MTBO vs. Optuna Comparative Analysis Report\n\n")
    f.write(f"**Execution Date:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("## 1. Executive Summary\n")
    f.write("This report benchmarks Multi-Task Bayesian Optimization (MTBO, Swersky et al. 2013) executed in a 4x25 batch architecture against dedicated single-game Optuna Tree-structured Parzen Estimator (TPE) studies across 100 strongly monotone quadratic games. The objective function is the Logarithmic Total Systemic Regret over 20,000 steps ($dt = 0.001$, $\\varepsilon = 10^{-4}$ fixed).\n\n")

    f.write("## 2. Computational Timing Benchmark & Extrapolation\n\n")
    f.write("| Algorithm | MTBO Actual Time (100 Games) | Optuna Time (5 Games) | Optuna Extrapolated (100 Games) | MTBO Speedup Factor |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- |\n")
    f.write(f"| **ZOGrandientPlay** | {t_zo:.1f} s ({t_zo/60:.2f} min) | {t_zo_optuna_5:.1f} s ({t_zo_optuna_5/60:.2f} min) | {t_zo_optuna_100_extrap:.1f} s ({t_zo_optuna_100_extrap/3600:.2f} h) | **{speedup_zo:.2f}x** |\n")
    f.write(f"| **NESeeking5D** | {t_ne:.1f} s ({t_ne/60:.2f} min) | {t_ne_optuna_5:.1f} s ({t_ne_optuna_5/60:.2f} min) | {t_ne_optuna_100_extrap:.1f} s ({t_ne_optuna_100_extrap/3600:.2f} h) | **{speedup_ne:.2f}x** |\n\n")

    f.write("### Evaluation Budgets & Early Stopping Summary:\n")
    f.write(f"- **ZOGrandientPlay MTBO (4x25):** Evaluated 4 batches with $n_{{{{\\text{{init}}}}}}=16$ per batch ({4*16*25} initial observations) + {zo_total_bo_steps} sequential BO steps across batches.\n")
    f.write(f"- **NESeeking5D MTBO (4x25):** Evaluated 4 batches with $n_{{{{\\text{{init}}}}}}=32$ per batch ({4*32*25} initial observations) + {ne_total_bo_steps} sequential BO steps across batches.\n\n")

    f.write("## 3. Representative Games Performance (Log10 Total Regret)\n\n")
    f.write("| Game | Category | True NE | ZO Optuna | ZO MTBO Tailored | ZO MTBO Consensus Default | NE Optuna | NE MTBO Tailored | NE MTBO Consensus Default |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |\n")
    for row in rep_rows:
        f.write(f"| Game {row['game_id']} | {row['category']} | {row['true_ne']} | {row['zo_optuna']:.4f} | {row['zo_mtbo_tailored']:.4f} | {row['zo_mtbo_default']:.4f} | {row['ne_optuna']:.4f} | {row['ne_mtbo_tailored']:.4f} | {row['ne_mtbo_default']:.4f} |\n")
    f.write("\n")

    f.write("## 4. Surrogate Model Validation & Uncertainty Calibration\n\n")
    f.write("| Algorithm | LOO RMSE | Standardized MSE (SMSE) | Calibration Std | Anomalies (Default > Tailored) | Surrogate Refinements |\n")
    f.write("| :--- | :--- | :--- | :--- | :--- | :--- |\n")
    f.write(f"| **ZOGrandientPlay** | {zo_val_metrics['loo_rmse']:.4f} | {zo_val_metrics['loo_smse']:.4f} | {zo_val_metrics['calibration_std']:.4f} | {zo_val_metrics['anomalies_detected']} | {zo_val_metrics['refinements_executed']} |\n")
    f.write(f"| **NESeeking5D** | {ne_val_metrics['loo_rmse']:.4f} | {ne_val_metrics['loo_smse']:.4f} | {ne_val_metrics['calibration_std']:.4f} | {ne_val_metrics['anomalies_detected']} | {ne_val_metrics['refinements_executed']} |\n\n")

    f.write("## 5. 100-Game Global Ensemble Statistics & Statistical Significance\n\n")
    f.write("| Metric | ZOGrandientPlay | NESeeking5D |\n")
    f.write("| :--- | :--- | :--- |\n")
    f.write(f"| **Tailored Optima (Mean +/- Std)** | {np.mean(zo_losses_opt):.4f} +/- {np.std(zo_losses_opt):.4f} | {np.mean(ne_losses_opt):.4f} +/- {np.std(ne_losses_opt):.4f} |\n")
    f.write(f"| **Tailored Optima (Median)** | {np.median(zo_losses_opt):.4f} | {np.median(ne_losses_opt):.4f} |\n")
    f.write(f"| **Consensus Default (Mean +/- Std)** | {np.mean(zo_losses_def):.4f} +/- {np.std(zo_losses_def):.4f} | {np.mean(ne_losses_def):.4f} +/- {np.std(ne_losses_def):.4f} |\n")
    f.write(f"| **Consensus Default (Median)** | {np.median(zo_losses_def):.4f} | {np.median(ne_losses_def):.4f} |\n")
    f.write(f"| **Mean Tunability Gain (Def - Tailored)** | {np.mean(zo_diff):.4f} | {np.mean(ne_diff):.4f} |\n")
    f.write(f"| **Tunability 95% Confidence Interval** | [{zo_ci95[0]:.4f}, {zo_ci95[1]:.4f}] | [{ne_ci95[0]:.4f}, {ne_ci95[1]:.4f}] |\n")
    f.write(f"| **Max Tunability Gain** | {np.max(zo_diff):.4f} | {np.max(ne_diff):.4f} |\n")
    f.write(f"| **Paired t-test (p-value)** | {zo_t_pval:.4e} | {ne_t_pval:.4e} |\n")
    f.write(f"| **Wilcoxon Signed-Rank (p-value)** | {zo_w_pval:.4e} | {ne_w_pval:.4e} |\n\n")

    f.write("## 6. Strategic Recommendation\n")
    if speedup_ne > 1.2 and np.mean(ne_diff) >= 0.0:
        f.write("- **Recommendation: RETAIN MTBO.** MTBO demonstrates superior computational efficiency through cross-task transfer learning, finding both game-specific optima and universal consensus defaults simultaneously while saving substantial compute compared to 100 independent Optuna runs.\n")
    else:
        f.write("- **Recommendation: RECONSIDER OPTUNA.** Direct per-game Optuna tuning showed competitive or superior runtime and solution quality without surrogate modeling overhead.\n")

print(f"Comprehensive morning report successfully generated at: {report_path}")


## 7. Hybrid Strategy: MTBO Consensus Default + Parallel Single-Task BO (100 Games)

The Hybrid Strategy decouples universal consensus default determination from dedicated per-game tailoring:
1. **Universal Consensus Default:** Inherited from the multi-task surrogate (Stage 1), where cross-task pooling across all 100 games completely eliminates stochastic noise and yields a variance-regularized, stable default parameter set.
2. **Parallel Single-Task BO:** Rather than fitting a single joint `MultiTaskGP` which suffers from cross-task negative transfer and cubic complexity $\mathcal{O}((M \cdot N)^3)$, we invoke [`run_parallel_single_task_bo()`](file:///home/bruno/Documents/PayoffBasedNEComparison/payoffnash/mtbo.py) across all games simultaneously.
   - Each game has an independent `SingleTaskGP` with continuous `LogExpectedImprovement` optimization.
   - Sample Average Approximation (SAA) with $K = 5$ Common Random Numbers seeds for `ZOGrandientPlay` eliminates lucky-noise overfitting.
   - Results and plots are co-located in dedicated folder `data/07_hybrid_parallel_bo_100_games/`.


In [ ]:
# 7.1 ZOGrandientPlay: Parallel Single-Task BO (All 100 Games, SAA K=5)
data_dir_07 = "data/07_hybrid_parallel_bo_100_games"
os.makedirs(data_dir_07, exist_ok=True)

print("=== Starting ZOGrandientPlay Parallel Single-Task BO (100 Independent GPs, SAA K=5) ===", flush=True)
t0_zo_par = time.time()

zo_par_opt = MetaParamOptimization(
    game_ensemble=games,
    search_space=zo_search_space,
    algorithm_class=ZOGrandientPlay,
    cost_function=quadratic_game_regret_cost,
    output_path=os.path.join(data_dir_07, "zo_parallel_results.csv"),
    num_steps=20000,
    n_init=64,
    max_bo_steps=200,
    min_bo_steps=20,
    sim_per_param=5,
    eps_ei=1e-5,
    patience=20,
    tol_stagnation=1e-5,
    seed=14062003,
    fixed_params=zo_fixed_params,
    scaled_params=zo_scaled_params,
)

zo_par_results = zo_par_opt.run_parallel_single_task_bo()
zo_parallel_thetas = zo_par_opt.best_thetas
t_zo_par = time.time() - t0_zo_par
print(f"ZOGrandientPlay Parallel Single-Task BO complete in {t_zo_par:.2f}s ({t_zo_par/60:.2f} min).", flush=True)

import gc
gc.collect()


In [ ]:
# 7.2 NESeeking5D: Parallel Single-Task BO (All 100 Games, 5D Search Space)
print("=== Starting NESeeking5D Parallel Single-Task BO (100 Independent GPs, 5D Continuous) ===", flush=True)
t0_ne_par = time.time()

ne_par_opt = MetaParamOptimization(
    game_ensemble=games,
    search_space=ne_search_space,
    algorithm_class=NESeeking5D,
    cost_function=quadratic_game_regret_cost,
    output_path=os.path.join(data_dir_07, "ne_parallel_results.csv"),
    num_steps=20000,
    n_init=64,
    max_bo_steps=200,
    min_bo_steps=20,
    sim_per_param=1,
    eps_ei=1e-5,
    patience=20,
    tol_stagnation=1e-5,
    seed=14062003,
    fixed_params=ne_fixed_params,
    scaled_params=ne_scaled_params,
)

ne_par_results = ne_par_opt.run_parallel_single_task_bo()
ne_parallel_thetas = ne_par_opt.best_thetas
t_ne_par = time.time() - t0_ne_par
print(f"NESeeking5D Parallel Single-Task BO complete in {t_ne_par:.2f}s ({t_ne_par/60:.2f} min).", flush=True)

import gc
gc.collect()


In [ ]:
# 7.3 Quantitative Hybrid Comparative Evaluation & Statistics (MTBO Default vs MTBO Tailored vs Parallel Single-Task)
print("--- Evaluating Parallel Single-Task Optima on All 100 Games in Parallel ---", flush=True)

# 1. ZOGP Parallel evaluations
zo_par_requests = [(zo_parallel_thetas[j], j) for j in range(len(games))]
zo_losses_par = np.array(zo_evaluator.evaluate_batch(zo_par_requests))

# 2. NESeeking Parallel evaluations
ne_par_requests = [(ne_parallel_thetas[j], j) for j in range(len(games))]
ne_losses_par = np.array(ne_evaluator.evaluate_batch(ne_par_requests))

# Tunability comparisons
zo_tunability_par = zo_losses_def - zo_losses_par
ne_tunability_par = ne_losses_def - ne_losses_par
zo_par_vs_mtbo = zo_losses_opt - zo_losses_par  # Positive means Parallel is better than MTBO tailored
ne_par_vs_mtbo = ne_losses_opt - ne_losses_par

print()
print("=" * 85)
print(f"{'Metric':<35} | {'ZOGrandientPlay':<22} | {'NESeeking5D':<22}")
print("-" * 85)
print(f"{'MTBO Consensus Default':<35} | {np.mean(zo_losses_def):.4f} +/- {np.std(zo_losses_def):.4f} | {np.mean(ne_losses_def):.4f} +/- {np.std(ne_losses_def):.4f}")
print(f"{'MTBO Tailored Optima':<35} | {np.mean(zo_losses_opt):.4f} +/- {np.std(zo_losses_opt):.4f} | {np.mean(ne_losses_opt):.4f} +/- {np.std(ne_losses_opt):.4f}")
print(f"{'Parallel Single-Task Optima':<35} | {np.mean(zo_losses_par):.4f} +/- {np.std(zo_losses_par):.4f} | {np.mean(ne_losses_par):.4f} +/- {np.std(ne_losses_par):.4f}")
print(f"{'Tunability Gain (Def - Par)':<35} | {np.mean(zo_tunability_par):.4f}               | {np.mean(ne_tunability_par):.4f}")
print(f"{'Gain over MTBO (MTBO - Par)':<35} | {np.mean(zo_par_vs_mtbo):.4f}               | {np.mean(ne_par_vs_mtbo):.4f}")
print("=" * 85)
print(flush=True)

# Save hybrid 100-game metrics CSV
hybrid_csv_path = os.path.join(data_dir_07, "hybrid_100_games_metrics.csv")
with open(hybrid_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["game_id", "zo_default", "zo_mtbo_tailored", "zo_parallel_tailored", "ne_default", "ne_mtbo_tailored", "ne_parallel_tailored"])
    for j in range(len(games)):
        writer.writerow([
            f"game_{j}",
            zo_losses_def[j], zo_losses_opt[j], zo_losses_par[j],
            ne_losses_def[j], ne_losses_opt[j], ne_losses_par[j],
        ])

import gc
gc.collect()


### 7.4 Dedicated Trajectory Visualizations on Representative Games

To avoid visual clutter, comparisons are split into two separate, clean 2x2 figures for each representative game:
- **Plot Set 1 (Optuna vs. Parallel Single-Task BO):** Contrasts dedicated per-game Optuna TPE with dedicated Parallel Single-Task BO.
- **Plot Set 2 (MTBO Tailored vs. Parallel Single-Task BO vs. Consensus Default):** Directly compares multi-task transfer learning against independent single-task modeling and the variance-regularized consensus default.


In [ ]:
# 7.5 Uncluttered 2-Way Comparative Plots for ZOGrandientPlay & NESeeking5D on 5 Representative Games
num_sim_steps = 20000

for g_idx, category, g in rep_games:
    x_star = g.nash_equilibrium.ravel()
    bounds = np.array([[iv[1] - iv[0] for iv in s] for s in g.action_sets], dtype=float)
    min_w = float(np.min(bounds))
    w0, w1 = float(bounds[0, 0]), float(bounds[1, 0])

    # ------------------ ZOGrandientPlay Simulations ------------------
    # 1. Parallel Single-Task BO
    est_zo_par = zo_par_opt.build_estimator(zo_parallel_thetas[g_idx], g)
    res_zo_par = est_zo_par.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_zo_par = quadratic_game_regret_cost(g, res_zo_par)

    # 2. MTBO Tailored
    est_zo_mtbo = zo_evaluator.build_estimator(zo_best_thetas[g_idx], g)
    res_zo_mtbo = est_zo_mtbo.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_zo_mtbo = quadratic_game_regret_cost(g, res_zo_mtbo)

    # 3. MTBO Consensus Default
    est_zo_def = zo_evaluator.build_estimator(zo_best_theta_def, g)
    res_zo_def = est_zo_def.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_zo_def = quadratic_game_regret_cost(g, res_zo_def)

    # 4. Optuna TPE
    zo_best_p = zo_optuna_results[g_idx][0]
    est_zo_tpe = ZOGrandientPlay(g, v=zo_best_p["v"], a=zo_best_p["a"] * min_w, epsilon=1e-4)
    res_zo_tpe = est_zo_tpe.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_zo_tpe = quadratic_game_regret_cost(g, res_zo_tpe)

    t_steps = np.arange(len(res_zo_par.action_history))

    # === ZOGP Plot Set 1: Optuna vs. Parallel Single-Task BO ===
    fig1, ax1 = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig1.suptitle(
        f"ZOGP Game {g_idx} ({category}) — Parallel Single-Task BO vs. Optuna TPE\n"
        f"Regret Loss: Parallel BO={cost_zo_par:.4f} | Optuna={cost_zo_tpe:.4f} | NE: [{x_star[0]:.3f}, {x_star[1]:.3f}]",
        fontsize=12, fontweight="bold"
    )
    ax1[0, 0].plot(t_steps, res_zo_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax1[0, 0].plot(t_steps, res_zo_tpe.action_history[:, 0], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna TPE ({cost_zo_tpe:.3f})")
    ax1[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax1[0, 0].set_title("Player 1: Parallel BO vs. Optuna"); ax1[0, 0].set_ylabel("Action u1"); ax1[0, 0].grid(True, alpha=0.3); ax1[0, 0].legend(loc="best")

    ax1[0, 1].plot(t_steps, res_zo_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax1[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax1[0, 1].set_title("Player 1: Parallel BO Trajectory"); ax1[0, 1].set_ylabel("Action u1"); ax1[0, 1].grid(True, alpha=0.3); ax1[0, 1].legend(loc="best")

    ax1[1, 0].plot(t_steps, res_zo_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax1[1, 0].plot(t_steps, res_zo_tpe.action_history[:, 1], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna TPE ({cost_zo_tpe:.3f})")
    ax1[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax1[1, 0].set_title("Player 2: Parallel BO vs. Optuna"); ax1[1, 0].set_xlabel("Iteration Step"); ax1[1, 0].set_ylabel("Action u2"); ax1[1, 0].grid(True, alpha=0.3); ax1[1, 0].legend(loc="best")

    ax1[1, 1].plot(t_steps, res_zo_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax1[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax1[1, 1].set_title("Player 2: Parallel BO Trajectory"); ax1[1, 1].set_xlabel("Iteration Step"); ax1[1, 1].set_ylabel("Action u2"); ax1[1, 1].grid(True, alpha=0.3); ax1[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir_07, f"zo_rep_game_{g_idx}_optuna_vs_parallel.png"), dpi=300)
    plt.show()
    plt.close(fig1)
    plt.close('all')

    # === ZOGP Plot Set 2: MTBO Tailored vs. Parallel Single-Task BO vs. MTBO Consensus Default ===
    fig2, ax2 = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig2.suptitle(
        f"ZOGP Game {g_idx} ({category}) — MTBO Tailored vs. Parallel Single-Task BO vs. Default\n"
        f"Regret Loss: Parallel BO={cost_zo_par:.4f} | MTBO Tailored={cost_zo_mtbo:.4f} | MTBO Consensus Default={cost_zo_def:.4f}",
        fontsize=12, fontweight="bold"
    )
    ax2[0, 0].plot(t_steps, res_zo_mtbo.action_history[:, 0], color="tab:blue", lw=1.2, label=f"MTBO Tailored ({cost_zo_mtbo:.3f})")
    ax2[0, 0].plot(t_steps, res_zo_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax2[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax2[0, 0].set_title("Player 1: MTBO vs. Parallel BO"); ax2[0, 0].set_ylabel("Action u1"); ax2[0, 0].grid(True, alpha=0.3); ax2[0, 0].legend(loc="best")

    ax2[0, 1].plot(t_steps, res_zo_def.action_history[:, 0], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default ({cost_zo_def:.3f})")
    ax2[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax2[0, 1].set_title("Player 1: Universal Consensus Default"); ax2[0, 1].set_ylabel("Action u1"); ax2[0, 1].grid(True, alpha=0.3); ax2[0, 1].legend(loc="best")

    ax2[1, 0].plot(t_steps, res_zo_mtbo.action_history[:, 1], color="tab:blue", lw=1.2, label=f"MTBO Tailored ({cost_zo_mtbo:.3f})")
    ax2[1, 0].plot(t_steps, res_zo_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_zo_par:.3f})")
    ax2[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax2[1, 0].set_title("Player 2: MTBO vs. Parallel BO"); ax2[1, 0].set_xlabel("Iteration Step"); ax2[1, 0].set_ylabel("Action u2"); ax2[1, 0].grid(True, alpha=0.3); ax2[1, 0].legend(loc="best")

    ax2[1, 1].plot(t_steps, res_zo_def.action_history[:, 1], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default ({cost_zo_def:.3f})")
    ax2[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax2[1, 1].set_title("Player 2: Universal Consensus Default"); ax2[1, 1].set_xlabel("Iteration Step"); ax2[1, 1].set_ylabel("Action u2"); ax2[1, 1].grid(True, alpha=0.3); ax2[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir_07, f"zo_rep_game_{g_idx}_mtbo_vs_parallel.png"), dpi=300)
    plt.show()
    plt.close(fig2)
    plt.close('all')

    # ------------------ NESeeking5D Simulations ------------------
    # 1. Parallel Single-Task BO
    est_ne_par = ne_par_opt.build_estimator(ne_parallel_thetas[g_idx], g)
    res_ne_par = est_ne_par.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_ne_par = quadratic_game_regret_cost(g, res_ne_par)

    # 2. MTBO Tailored
    est_ne_mtbo = ne_evaluator.build_estimator(ne_best_thetas[g_idx], g)
    res_ne_mtbo = est_ne_mtbo.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_ne_mtbo = quadratic_game_regret_cost(g, res_ne_mtbo)

    # 3. MTBO Consensus Default
    est_ne_def = ne_evaluator.build_estimator(ne_best_theta_def, g)
    res_ne_def = est_ne_def.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_ne_def = quadratic_game_regret_cost(g, res_ne_def)

    # 4. Optuna TPE
    ne_best_p = ne_optuna_results[g_idx][0]
    est_ne_tpe = NESeeking5D(
        g,
        k=np.array([[ne_best_p["k[0,0]"]], [ne_best_p["k[1,0]"]]]),
        a=np.array([[ne_best_p["a[0,0]"] * w0], [ne_best_p["a[1,0]"] * w1]]),
        omega1=ne_best_p["omega1"],
        dt=0.001,
    )
    res_ne_tpe = est_ne_tpe.estimate(np.zeros((g.N, 1)), num_steps=num_sim_steps)
    cost_ne_tpe = quadratic_game_regret_cost(g, res_ne_tpe)

    t_ne_steps = np.arange(len(res_ne_par.action_history)) * 0.001

    # === NESeeking Plot Set 1: Optuna vs. Parallel Single-Task BO ===
    fig3, ax3 = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig3.suptitle(
        f"NESeeking5D Game {g_idx} ({category}) — Parallel Single-Task BO vs. Optuna TPE\n"
        f"Regret Loss: Parallel BO={cost_ne_par:.4f} | Optuna={cost_ne_tpe:.4f} | NE: [{x_star[0]:.3f}, {x_star[1]:.3f}]",
        fontsize=12, fontweight="bold"
    )
    ax3[0, 0].plot(t_ne_steps, res_ne_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax3[0, 0].plot(t_ne_steps, res_ne_tpe.action_history[:, 0], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna TPE ({cost_ne_tpe:.3f})")
    ax3[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax3[0, 0].set_title("Player 1: Parallel BO vs. Optuna"); ax3[0, 0].set_ylabel("Action u1"); ax3[0, 0].grid(True, alpha=0.3); ax3[0, 0].legend(loc="best")

    ax3[0, 1].plot(t_ne_steps, res_ne_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax3[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax3[0, 1].set_title("Player 1: Parallel BO Trajectory"); ax3[0, 1].set_ylabel("Action u1"); ax3[0, 1].grid(True, alpha=0.3); ax3[0, 1].legend(loc="best")

    ax3[1, 0].plot(t_ne_steps, res_ne_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax3[1, 0].plot(t_ne_steps, res_ne_tpe.action_history[:, 1], color="tab:green", lw=1.0, alpha=0.8, label=f"Optuna TPE ({cost_ne_tpe:.3f})")
    ax3[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax3[1, 0].set_title("Player 2: Parallel BO vs. Optuna"); ax3[1, 0].set_xlabel("Time (s)"); ax3[1, 0].set_ylabel("Action u2"); ax3[1, 0].grid(True, alpha=0.3); ax3[1, 0].legend(loc="best")

    ax3[1, 1].plot(t_ne_steps, res_ne_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax3[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax3[1, 1].set_title("Player 2: Parallel BO Trajectory"); ax3[1, 1].set_xlabel("Time (s)"); ax3[1, 1].set_ylabel("Action u2"); ax3[1, 1].grid(True, alpha=0.3); ax3[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir_07, f"ne_rep_game_{g_idx}_optuna_vs_parallel.png"), dpi=300)
    plt.show()
    plt.close(fig3)
    plt.close('all')

    # === NESeeking Plot Set 2: MTBO Tailored vs. Parallel Single-Task BO vs. MTBO Consensus Default ===
    fig4, ax4 = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig4.suptitle(
        f"NESeeking5D Game {g_idx} ({category}) — MTBO Tailored vs. Parallel Single-Task BO vs. Default\n"
        f"Regret Loss: Parallel BO={cost_ne_par:.4f} | MTBO Tailored={cost_ne_mtbo:.4f} | MTBO Consensus Default={cost_ne_def:.4f}",
        fontsize=12, fontweight="bold"
    )
    ax4[0, 0].plot(t_ne_steps, res_ne_mtbo.action_history[:, 0], color="tab:blue", lw=1.2, label=f"MTBO Tailored ({cost_ne_mtbo:.3f})")
    ax4[0, 0].plot(t_ne_steps, res_ne_par.action_history[:, 0], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax4[0, 0].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax4[0, 0].set_title("Player 1: MTBO vs. Parallel BO"); ax4[0, 0].set_ylabel("Action u1"); ax4[0, 0].grid(True, alpha=0.3); ax4[0, 0].legend(loc="best")

    ax4[0, 1].plot(t_ne_steps, res_ne_def.action_history[:, 0], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default ({cost_ne_def:.3f})")
    ax4[0, 1].axhline(x_star[0], color="black", linestyle="--", lw=1.5, label="True u1*")
    ax4[0, 1].set_title("Player 1: Universal Consensus Default"); ax4[0, 1].set_ylabel("Action u1"); ax4[0, 1].grid(True, alpha=0.3); ax4[0, 1].legend(loc="best")

    ax4[1, 0].plot(t_ne_steps, res_ne_mtbo.action_history[:, 1], color="tab:blue", lw=1.2, label=f"MTBO Tailored ({cost_ne_mtbo:.3f})")
    ax4[1, 0].plot(t_ne_steps, res_ne_par.action_history[:, 1], color="tab:purple", lw=1.2, label=f"Parallel SingleTask BO ({cost_ne_par:.3f})")
    ax4[1, 0].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax4[1, 0].set_title("Player 2: MTBO vs. Parallel BO"); ax4[1, 0].set_xlabel("Time (s)"); ax4[1, 0].set_ylabel("Action u2"); ax4[1, 0].grid(True, alpha=0.3); ax4[1, 0].legend(loc="best")

    ax4[1, 1].plot(t_ne_steps, res_ne_def.action_history[:, 1], color="tab:orange", lw=1.2, label=f"MTBO Consensus Default ({cost_ne_def:.3f})")
    ax4[1, 1].axhline(x_star[1], color="black", linestyle="--", lw=1.5, label="True u2*")
    ax4[1, 1].set_title("Player 2: Universal Consensus Default"); ax4[1, 1].set_xlabel("Time (s)"); ax4[1, 1].set_ylabel("Action u2"); ax4[1, 1].grid(True, alpha=0.3); ax4[1, 1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(os.path.join(data_dir_07, f"ne_rep_game_{g_idx}_mtbo_vs_parallel.png"), dpi=300)
    plt.show()
    plt.close(fig4)
    plt.close('all')

    import gc
    gc.collect()

print("All uncluttered representative plots saved successfully in:", data_dir_07, flush=True)
